# Tokenizers in Machine Learning: Organized Starter Guide

A practical notebook to understand how tokenization decisions impact real ML systems, from data preparation to model serving.

## 1) Tokenization Concept Map (Simple + Organized)

Tokenization is the process of converting raw text into smaller units that ML models can consume.

### End-to-End Flow

```text
Raw text
  -> Normalize (unicode, case-fold, cleanup)
  -> Tokenize (word/subword/byte/char)
  -> Convert to IDs (vocabulary lookup)
  -> Length handling (trim/chunk/pad)
  -> Model input tensors (ids, masks, type ids)
```

### Where It Is Used in Real Time

- Support automation: classify incoming tickets into billing/technical/shipping queues.
- Search systems: split and normalize user queries for better matching.
- Spam and abuse detection: detect suspicious token patterns from live messages.
- Translation and multilingual NLP: use subword units so unseen words remain processable.
- Mobile NLP: run fast tokenization on-device for low latency and privacy.

### Special Do and Don't at the Start

**Do**
- Keep the same preprocessing pipeline for training and inference.
- Choose tokenizer type based on task and language coverage.
- Track unknown-token behavior (OOV) during validation.

**Don't**
- Don't assume whitespace tokenization works well for every language.
- Don't change tokenizer or vocabulary without versioning model artifacts.
- Don't pad first and then trim; trim first for efficiency and cleaner inputs.

### Why This Matters

Models learn from token IDs, not raw strings. Better tokenization usually means better accuracy, stability, and production reliability.

In [1]:
import tensorflow as tf
import tensorflow_text as tf_text

print('TensorFlow version:', tf.__version__)

# Real-time use case: route support messages by intent keywords.
# Step 1: tokenize live messages.
sample_texts = tf.constant([
    'Need refund for duplicate payment',
    'App crashes after latest update',
    'Where is my order tracking id?'
])

ws_tokenizer = tf_text.WhitespaceTokenizer()
tokens = ws_tokenizer.tokenize(sample_texts)

print('\nInput messages:')
for text in sample_texts.numpy():
    print(' -', text.decode('utf-8'))

print('\nWhitespace tokens:')
for text, row in zip(sample_texts.numpy(), tokens.to_list()):
    print(text.decode('utf-8'), '->', [t.decode('utf-8') for t in row])

# Step 2: lightweight keyword mapping demo (teaching purpose only).
intent_keywords = {
    'billing': {'refund', 'payment'},
    'technical': {'crashes', 'update', 'app'},
    'shipping': {'order', 'tracking'},
}

def route_intent(token_row):
    words = {t.decode('utf-8').lower() for t in token_row}
    scores = {k: len(words & v) for k, v in intent_keywords.items()}
    return max(scores, key=scores.get) if max(scores.values()) > 0 else 'other'

print('\nSimple routing example:')
for text, row in zip(sample_texts.numpy(), tokens.to_list()):
    print(f"{text.decode('utf-8')} -> {route_intent(row)}")

TensorFlow version: 2.22.0-dev0+selfbuilt

Input messages:
 - Need refund for duplicate payment
 - App crashes after latest update
 - Where is my order tracking id?

Whitespace tokens:
Need refund for duplicate payment -> ['Need', 'refund', 'for', 'duplicate', 'payment']
App crashes after latest update -> ['App', 'crashes', 'after', 'latest', 'update']
Where is my order tracking id? -> ['Where', 'is', 'my', 'order', 'tracking', 'id?']

Simple routing example:
Need refund for duplicate payment -> billing
App crashes after latest update -> technical
Where is my order tracking id? -> shipping


In [ ]:
import importlib.util
import os


def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), "configs", "runtime.env"),
        os.path.join(os.getcwd(), "configs", "runtime.env.example"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env.example"),
    ]
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip("\"'")
                if key and key not in os.environ:
                    os.environ[key] = value
        break


def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}


load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))
NO_CUDA = not USE_GPU

_torch_cuda = False
_tf_gpu = False
if importlib.util.find_spec("torch") is not None:
    import torch

    _torch_cuda = torch.cuda.is_available()
if importlib.util.find_spec("tensorflow") is not None:
    import tensorflow as tf

    _tf_gpu = bool(tf.config.list_physical_devices("GPU"))

RUNTIME_DEVICE = "cuda" if USE_GPU and (_torch_cuda or _tf_gpu) else "cpu"
print(f"USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}")

## 1A) Certification-Aligned ML Mapping (NVIDIA NCA-GENL / NCP-GENL Context)

This section maps tokenizer concepts to common Generative AI certification themes and real implementation tasks.

### Topic Map: Concept -> Why It Matters -> Tokenizer Example

- **Data Preparation for LLM Pipelines**
  - Why: Input consistency improves model stability and retrieval quality.
  - Tokenizer link: `normalize_utf8`, `case_fold_utf8`, `WhitespaceTokenizer`, `UnicodeScriptTokenizer`.
  - Example: normalize multilingual support tickets before intent classification.

- **Prompt and Context Window Engineering**
  - Why: LLMs are context-limited; token budget must be managed.
  - Tokenizer link: `ShrinkLongestTrimmer`, `RoundRobinTrimmer`, `WaterfallTrimmer`, `sliding_window`.
  - Example: QA pair packing as `[question, context]` under max length.

- **RAG Pipeline Readiness**
  - Why: retrieval quality depends on chunking/token consistency.
  - Tokenizer link: Sentence/subword tokenization + overlap chunking.
  - Example: chunk policy docs with overlap to avoid cutting key entities.

- **Evaluation and Operations (MLOps/KPI)**
  - Why: production systems need measurable throughput, latency, and quality retention.
  - Tokenizer link: benchmark trimmers and token retention ratio.
  - Example KPI: p95 latency, tokens/sec, retained-token ratio after trimming.

- **Safety and Enterprise Controls**
  - Why: bad token handling can leak context or break moderation flows.
  - Tokenizer link: Unicode normalization + robust OOV handling.
  - Example: sanitize malformed UTF-8 and keep deterministic preprocessing across environments.

### Quick Exam-Style Takeaway Points

- Tokenization is not only preprocessing; it is a **cost, latency, and quality control layer**.
- Trimming strategy impacts **fairness between segments** (question vs context, query vs document).
- KPI tracking should include **speed and semantic retention**, not speed alone.

### `tensorflow_text`
Graph-native tokenizers that export with the model.

#### Subword Tokenizers
- `text.BertTokenizer`
- `text.FastBertTokenizer`
- `text.FastSentencepieceTokenizer`
- `text.FastWordpieceTokenizer`

#### Segmentation Tokenizers
- `text.WhitespaceTokenizer`
- `text.UnicodeScriptTokenizer`
- `text.PhraseTokenizer`

#### Low-Level
- `text.ByteSplitter`
- `text.StateBasedSentenceBreaker`

Below are compact examples in separate cells for each tokenizer, with both common and edge-case inputs.

In [2]:
# Trimmer selection by use case + KPI benchmark
import time

print('=== Trimmer Use-Case Guide ===')
print('ShrinkLongest  : Best when one segment is much longer and both segments matter.')
print('RoundRobin     : Best when both segments should be preserved as fairly as possible.')
print('Waterfall      : Best when segment priority is explicit (e.g., keep query fully, trim context first).')

# Sample two-segment workload (query, context)
batch_size = 128
queries = tf.constant([
    'refund failed transaction issue urgent support',
    'account verification blocked after password reset',
    'where is shipment tracking link for order',
    'model latency increased after deployment patch',
] * (batch_size // 4))

contexts = tf.constant([
    'customer requested refund because duplicate charge appeared in monthly statement and payment status remained pending',
    'user cannot verify account due to repeated otp mismatch while device timezone changed during travel',
    'shipment moved between hubs but final-mile carrier did not update tracking portal for several days',
    'service response time increased after new release and autoscaling policy did not trigger in time',
] * (batch_size // 4))

tok = tf_text.WhitespaceTokenizer()
seg_a = tok.tokenize(queries)
seg_b = tok.tokenize(contexts)

orig_a = tf.cast(seg_a.row_lengths(), tf.float32)
orig_b = tf.cast(seg_b.row_lengths(), tf.float32)
orig_total = tf.maximum(orig_a + orig_b, 1.0)

max_len = 20
trimmers = {
    'ShrinkLongest': tf_text.ShrinkLongestTrimmer(max_seq_length=max_len, axis=1),
    'RoundRobin': tf_text.RoundRobinTrimmer(max_seq_length=max_len, axis=1),
    'Waterfall': tf_text.WaterfallTrimmer(max_seq_length=max_len, axis=1),
}

def evaluate_trimmer(name, trimmer, iterations=60):
    # Warm-up
    out_a, out_b = trimmer.trim([seg_a, seg_b])

    start = time.perf_counter()
    for _ in range(iterations):
        out_a, out_b = trimmer.trim([seg_a, seg_b])
    end = time.perf_counter()

    elapsed = end - start
    ms_per_batch = (elapsed / iterations) * 1000.0
    throughput_batches_per_sec = iterations / elapsed
    throughput_rows_per_sec = (iterations * batch_size) / elapsed

    new_a = tf.cast(out_a.row_lengths(), tf.float32)
    new_b = tf.cast(out_b.row_lengths(), tf.float32)
    new_total = tf.maximum(new_a + new_b, 1.0)

    # KPI 1: Token retention ratio
    retention = tf.reduce_mean(new_total / orig_total).numpy()

    # KPI 2: Query preservation ratio (segment A priority KPI)
    query_preservation = tf.reduce_mean(new_a / tf.maximum(orig_a, 1.0)).numpy()

    # KPI 3: Fairness (higher means more balanced retention between A and B)
    fairness = (1.0 - tf.reduce_mean(tf.abs(new_a - new_b) / new_total)).numpy()

    print(f'\n{name}')
    print(f'  Latency (ms/batch):      {ms_per_batch:.4f}')
    print(f'  Throughput (batches/s):  {throughput_batches_per_sec:.2f}')
    print(f'  Throughput (rows/s):     {throughput_rows_per_sec:.0f}')
    print(f'  Retention ratio:         {retention:.4f}')
    print(f'  Query preservation:      {query_preservation:.4f}')
    print(f'  Fairness score:          {fairness:.4f}')

for n, t in trimmers.items():
    evaluate_trimmer(n, t)

print('\n=== Practical selection hint ===')
print('- Prefer ShrinkLongest when long context dominates and you need balanced signal retention.')
print('- Prefer RoundRobin when query and context should both survive proportionally.')
print('- Prefer Waterfall when segment A is business-critical and must be preserved first.')

=== Trimmer Use-Case Guide ===
ShrinkLongest  : Best when one segment is much longer and both segments matter.
RoundRobin     : Best when both segments should be preserved as fairly as possible.
Waterfall      : Best when segment priority is explicit (e.g., keep query fully, trim context first).

ShrinkLongest
  Latency (ms/batch):      72.0787
  Throughput (batches/s):  13.87
  Throughput (rows/s):     1776
  Retention ratio:         0.9416
  Query preservation:      1.0000
  Fairness score:          0.6250

RoundRobin
  Latency (ms/batch):      4.9134
  Throughput (batches/s):  203.53
  Throughput (rows/s):     26051
  Retention ratio:         0.9416
  Query preservation:      1.0000
  Fairness score:          0.6250

Waterfall
  Latency (ms/batch):      66.5081
  Throughput (batches/s):  15.04
  Throughput (rows/s):     1925
  Retention ratio:         0.9416
  Query preservation:      1.0000
  Fairness score:          0.6250

=== Practical selection hint ===
- Prefer ShrinkLongest w

In [3]:
# Standalone coding-assistance tokenizer comparison
code_samples = tf.constant([
    "def add_numbers(a, b): return a + b",
    "if user_id not in cache_map: cache_map[user_id] = compute_hash(user_id)",
    "for i in range(10): print(i, end=' | ')",
    "SELECT user_id, total$ FROM sales_2026 WHERE total$ > 1000;",
])

ws_tok = tf_text.WhitespaceTokenizer()
unicode_tok = tf_text.UnicodeScriptTokenizer()
byte_tok = tf_text.ByteSplitter()

ws_out = ws_tok.tokenize(code_samples).to_list()
unicode_out = unicode_tok.tokenize(code_samples).to_list()
byte_out = byte_tok.split(code_samples).to_list()

print('=== Code Tokenization Comparison ===')
for i, raw in enumerate(code_samples.numpy()):
    s = raw.decode('utf-8')
    ws_tokens = [t.decode('utf-8') for t in ws_out[i]]
    uni_tokens = [t.decode('utf-8') for t in unicode_out[i]]
    byte_count = len(byte_out[i])

    print(f"\nSample {i+1}: {s}")
    print(f"WhitespaceTokenizer tokens ({len(ws_tokens)}): {ws_tokens}")
    print(f"UnicodeScriptTokenizer tokens ({len(uni_tokens)}): {uni_tokens}")
    print(f"ByteSplitter units: {byte_count} bytes")

print('\n=== Recommendation for Coding Assistance ===')
print('1) Best coverage: Byte-level tokenization (no unknown symbol risk).')
print('2) Best practical balance: Subword tokenization (WordPiece/BPE/SentencePiece).')
print('3) Baseline only: Whitespace tokenization (fast but misses symbol granularity).')

print('\nWhen to use which:')
print('- Byte-level: minifying/parsing unusual symbols, mixed languages, robust fallback.')
print('- Subword (WordPiece/BPE/SentencePiece): code completion, refactor suggestions, bug fixing.')
print('- Whitespace: simple rule-based routing or quick prototypes.')

=== Code Tokenization Comparison ===

Sample 1: def add_numbers(a, b): return a + b
WhitespaceTokenizer tokens (7): ['def', 'add_numbers(a,', 'b):', 'return', 'a', '+', 'b']
UnicodeScriptTokenizer tokens (13): ['def', 'add', '_', 'numbers', '(', 'a', ',', 'b', '):', 'return', 'a', '+', 'b']
ByteSplitter units: 35 bytes

Sample 2: if user_id not in cache_map: cache_map[user_id] = compute_hash(user_id)
WhitespaceTokenizer tokens (8): ['if', 'user_id', 'not', 'in', 'cache_map:', 'cache_map[user_id]', '=', 'compute_hash(user_id)']
UnicodeScriptTokenizer tokens (26): ['if', 'user', '_', 'id', 'not', 'in', 'cache', '_', 'map', ':', 'cache', '_', 'map', '[', 'user', '_', 'id', ']=', 'compute', '_', 'hash', '(', 'user', '_', 'id', ')']
ByteSplitter units: 71 bytes

Sample 3: for i in range(10): print(i, end=' | ')
WhitespaceTokenizer tokens (8): ['for', 'i', 'in', 'range(10):', 'print(i,', "end='", '|', "')"]
UnicodeScriptTokenizer tokens (11): ['for', 'i', 'in', 'range', '(10):', 'print', '('

## 1B) Separate Cell: Which Tokenizer Helps Coding Assistance?

This standalone section compares tokenizers on code-like text and highlights when each one is useful for coding assistants.

### Practical conclusion
- For coding assistance, **byte-level or subword tokenization** is usually strongest.
- Byte-level coverage avoids unknown tokens for operators, indentation, and rare identifiers.
- Subword tokenization compresses frequent code patterns better than pure character/byte splitting.

## 1C) Expanded Use-Case Map: Tokenizer and API Concepts in This Notebook

This matrix lists high-value practical use cases supported by the APIs already demonstrated here.

| API / Concept | Best Use Cases | Production Notes |
|---|---|---|
| WhitespaceTokenizer | Quick baseline NLP, rule-based routing, simple English preprocessing | Fast and simple, but weak for punctuation-heavy or multilingual text |
| UnicodeScriptTokenizer | Mixed-script text, multilingual chat moderation, social stream analysis | Better script boundaries than whitespace splitting |
| ByteSplitter | Symbol-heavy/code-like text, robust fallback for unknown character patterns | Strong coverage, but sequence length can increase |
| PhraseTokenizer | Domain phrase capture (finance, medical, legal terms) | Keep vocabulary curated and versioned |
| BertTokenizer / WordpieceTokenizer | Classification, NER, search reranking, QA with BERT-style models | Ensure tokenizer matches pretrained model vocabulary |
| FastBertTokenizer / FastWordpieceTokenizer | Real-time serving, mobile inference, lower-latency preprocessing | Prefer for deployment and export-friendly pipelines |
| SentencepieceTokenizer / FastSentencepieceTokenizer | Multilingual and subword-rich corpora, noisy user text | Fast variant for export paths; standard variant for richer features |
| StateBasedSentenceBreaker | Document sentence segmentation, preprocessing for summarization/QA | Good before chunking and retrieval pipelines |
| RegexSplitter / regex_split | CSV/log parsing, delimiter-driven tokenization, structured text | Validate regex for empty-token behavior |
| normalize_utf8 / case_fold_utf8 | Unicode normalization, canonical text for consistent token hits | Apply before tokenization for train/serve parity |
| coerce_to_structurally_valid_utf8 | Repair malformed text streams from ingestion pipelines | Important in ETL and real-time ingestion |
| ShrinkLongestTrimmer | Balanced preservation when one segment is much longer | Common default for pair inputs under budget |
| RoundRobinTrimmer | Fair retention across segments in pair tasks | Useful when both segments are equally important |
| WaterfallTrimmer | Priority preservation (query-first or policy-first contexts) | Good for strict business-priority segments |
| trim_model_inputs | Hard cap sequence length before model input | Use before padding to control memory/latency |
| pad_model_inputs | Build dense tensors plus masks for transformer inputs | Validate masks and padding IDs |
| sliding_window | Long-document QA, compliance scanning, retrieval chunking | Use overlap to avoid boundary loss |
| ngrams / Reduction | Text classification features, lexical similarity, shallow NLP baselines | Combine with embeddings or classic feature models |
| BOISE tags + span utilities | NER extraction and span alignment workflows | Preserve offsets and decode constraints carefully |
| Item selectors / MLM masking concepts | Pretraining-style masked LM data pipelines | Track masking ratios and reproducibility |
| concatenate_segments / combine_segments | Pair input preparation for entailment, QA, semantic matching | Verify token_type_ids and [CLS]/[SEP] placement |
| max_spanning_tree / constrained decoding | Structured NLP prediction (dependency parsing/tag consistency) | Use when output validity constraints matter |

### KPI Suggestions by Use Case

- Real-time inference: p95 latency, throughput rows/s, memory footprint.
- Retrieval/QA: token retention ratio, query preservation ratio, answer-quality metric.
- NER/span tasks: exact span F1, offset consistency error rate.
- Long-doc pipelines: chunk coverage rate, boundary entity loss rate.
- Production reliability: OOV rate, tokenizer drift, train/serve parity checks.

## 1D) Interview + Certification + ML Practice Questions

Use these questions to test concept clarity from beginner to production-level understanding.

### A) Interview Questions (with short expected points)

1. What is the difference between tokenization, normalization, and vocabulary lookup?
- Tokenization splits text into units.
- Normalization standardizes text form (case, Unicode).
- Lookup maps tokens to numeric IDs.

2. Why are ragged tensors preferred before padding in NLP pipelines?
- Real text has variable length.
- Ragged tensors avoid memory waste from early padding.
- Trim and pad should happen close to model input creation.

3. When should you choose subword tokenization over whitespace tokenization?
- OOV-heavy domains.
- Multilingual or morphologically rich languages.
- Code and user-generated noisy text.

4. Why does sequence trimming strategy matter for model quality?
- It changes which context survives.
- Query/context balance can affect answer correctness.
- Different tasks need different preservation priorities.

5. Why should normalization happen before tokenization?
- Improves token consistency.
- Better vocabulary hit rate.
- Prevents offset mismatch and duplicate forms.

6. Why are Fast tokenizers preferred for deployment?
- Better export behavior in production graphs.
- Lower serving latency on edge/mobile workflows.
- Less runtime dependency complexity.

### B) Certification-Style MCQs (NCA-GENL/NCP-GENL aligned)

1. For multilingual support chats with mixed scripts, best default tokenizer choice is:
- A. Whitespace only
- B. UnicodeScriptTokenizer or SentencePiece
- C. Character blacklist splitter
- D. Static regex only

Answer: B

2. For pair tasks where segment A (question) must be preserved first, which trimmer is usually best?
- A. RoundRobinTrimmer
- B. WaterfallTrimmer
- C. ShrinkLongestTrimmer
- D. trim_model_inputs

Answer: B

3. Which KPI combination best validates tokenizer readiness for production?
- A. Only training accuracy
- B. Only vocabulary size
- C. p95 latency + throughput + token retention + quality metric
- D. Number of tokenizer classes

Answer: C

4. Which is the safest guidance for train/serve consistency?
- A. Different normalization for inference
- B. Shared preprocessing contract and versioned tokenizer/vocab
- C. Retrain tokenizer each deployment
- D. Skip attention masks

Answer: B

### C) ML Scenario Questions (concept-to-decision)

1. Scenario: Long legal document QA. Which APIs help most and why?
- `sliding_window` for chunking.
- `ShrinkLongestTrimmer` or `WaterfallTrimmer` for budget control.
- `pad_model_inputs` for dense model tensors.

2. Scenario: NER with accented text where offsets must map back to original.
- Use `normalize_utf8_with_offsets_map` pattern.
- Keep normalization and span mapping aligned.
- Decode with constrained sequence strategy when needed.

3. Scenario: Mobile inference with strict latency budget.
- Prefer Fast tokenizer variants.
- Track p95 latency and throughput.
- Keep model + tokenizer artifacts version-locked.

4. Scenario: Query-document retrieval mismatch in production.
- Verify same tokenizer and normalization on index/query sides.
- Audit OOV rate and drift.
- Re-check trimmer impact on query retention.

## 1E) Enterprise-Organized Learning Roadmap

Use this as a structured progression from concept learning to production ownership.

### Track 1: Foundation (Interview Readiness)

- Goal: explain tokenizer decisions clearly in interviews.
- Focus APIs: `WhitespaceTokenizer`, `UnicodeScriptTokenizer`, `ByteSplitter`, `WordpieceTokenizer`.
- Deliverable: explain trade-offs for accuracy, speed, and robustness.

### Track 2: Certification (NCA-GENL / NCP-GENL Style)

- Goal: choose correct preprocessing strategy for given constraints.
- Focus APIs: normalization (`normalize_utf8`, `case_fold_utf8`), trimmers, padding, masks.
- Deliverable: select best pipeline under latency/context constraints.

### Track 3: Enterprise Execution

- Goal: operate tokenizer pipelines in production safely.
- Focus APIs: Fast tokenizers, trimmers, chunking, span alignment, masking workflow.
- Deliverable: production checklist + KPI dashboard + rollback strategy.

### Enterprise Do and Don't

**Do**
- Standardize train/serve preprocessing contract and version it.
- Measure p50/p95 latency, throughput, retention ratio, and OOV drift.
- Use canary release for tokenizer/vocab changes.

**Don't**
- Don't switch tokenizer/vocab without revalidating model quality.
- Don't skip offset checks in NER/span-sensitive systems.
- Don't optimize only speed and ignore semantic retention.

## 1F) Enterprise Mock Assessment (Timed, Organized)

**Format:** 20 questions, 30 minutes, mixed interview + certification + ML operations.

### Part A: Concept and Design (8 questions)

1. Compare whitespace, subword, and byte-level tokenization for code + multilingual text.
2. Why must normalization be consistent between training and inference?
3. Explain when `RoundRobinTrimmer` is better than `WaterfallTrimmer`.
4. Why is `trim -> pad` preferred over `pad -> trim`?
5. What risks appear when tokenizer and model vocab versions are mismatched?
6. When should you preserve offset maps and why?
7. How do attention masks affect padded transformer inference?
8. Why are Fast tokenizer variants preferred for deployment paths?

### Part B: Certification-Style MCQ (6 questions, with immediate explanations)

9. Best tokenizer for mixed-script user chat streams?
- A) Whitespace only
- B) UnicodeScript/SentencePiece
- C) Character blacklist
- D) Regex only

**Correct: B**
- Why B is correct: script-aware and subword tokenizers handle multilingual boundaries and unknown forms better.
- Why A is not correct: whitespace fails for scripts without clear spaces and mixed-script boundaries.
- Why C is not correct: blacklisting is brittle and not a tokenizer strategy for multilingual generalization.
- Why D is not correct: regex-only pipelines are hard to maintain and weak for open-ended language variation.

10. Query-first pair task should prefer which trimming policy?
- A) RoundRobin
- B) Waterfall
- C) ShrinkLongest
- D) No trimmer

**Correct: B**
- Why B is correct: Waterfall preserves earlier segments first, ideal when query tokens are highest priority.
- Why A is not correct: RoundRobin favors fairness, not strict query-first preservation.
- Why C is not correct: ShrinkLongest balances by length, which can still trim critical query tokens in some layouts.
- Why D is not correct: no trimmer risks context overflow and model input violations.

11. Most complete readiness KPI set is:
- A) Accuracy only
- B) Latency only
- C) Latency + throughput + retention + quality
- D) Vocab size only

**Correct: C**
- Why C is correct: production readiness needs both system performance and semantic quality retention.
- Why A is not correct: accuracy alone ignores runtime feasibility and serving reliability.
- Why B is not correct: speed alone can hide quality degradation.
- Why D is not correct: vocab size is descriptive, not an outcome KPI.

12. For long-document QA, key API pattern is:
- A) Single pass
- B) sliding_window + overlap
- C) pad only
- D) regex only

**Correct: B**
- Why B is correct: overlap chunking preserves context continuity and reduces boundary misses.
- Why A is not correct: single pass often exceeds context window limits.
- Why C is not correct: padding does not solve over-length context segmentation.
- Why D is not correct: regex helps splitting but not robust context-window management by itself.

13. For span-sensitive NER after normalization, required practice is:
- A) Ignore offsets
- B) Keep offset mapping
- C) Remove diacritics always
- D) Disable trimmer

**Correct: B**
- Why B is correct: offset mapping is required to align predicted spans back to original text reliably.
- Why A is not correct: ignoring offsets breaks traceability for entities and annotations.
- Why C is not correct: always removing diacritics can damage meaning and language fidelity.
- Why D is not correct: trimmer control is unrelated to core offset-alignment requirement.

14. For mobile deployment, best tokenizer choice is usually:
- A) Python-only tokenizer path
- B) Fast tokenizer variants
- C) Regex splitter
- D) Char tokenizer only

**Correct: B**
- Why B is correct: Fast variants are generally better suited for deployment efficiency and portability.
- Why A is not correct: Python-only paths increase runtime dependency and deployment friction.
- Why C is not correct: regex alone is not a complete tokenizer solution for broad language behavior.
- Why D is not correct: char-only tokenization can be too long/inefficient for many mobile workloads.

### Part C: Enterprise Scenario (6 questions)

15. You see rising OOV rate in production. What are the first 3 checks?
16. p95 latency regresses after tokenizer change. What rollback and mitigation steps do you take?
17. Retrieval quality drops after index refresh. How do you verify tokenizer parity between query and index?
18. NER spans drift after normalization update. What validation suite do you run?
19. You must preserve question tokens under tight budget. Which trimmer and KPI prove success?
20. Which release strategy reduces tokenizer-risk best for enterprise systems?

### Enterprise Scoring Guide

- 18-20: Production-ready decision maker
- 15-17: Strong, minor gaps in ops rigor
- 12-14: Good fundamentals, needs deployment depth
- <12: Revisit tokenizer-trimmer-KPI alignment

## 1H) Model Answer Sheet (Enterprise Scenario Questions)

Use these as interview-ready, certification-friendly reference answers for Q15-Q20.

### Q15) OOV rate is rising in production. What are the first 3 checks?

1. **Tokenizer/version parity check**
- Confirm query pipeline, batch jobs, and model service are using the same tokenizer and vocab version.

2. **Data drift diagnosis**
- Compare current token distribution with training baseline.
- Inspect new domains, slang, code-mixed text, and language mix shift.

3. **Preprocessing contract validation**
- Verify normalization order and settings are unchanged between train and serve.
- Audit case-fold, Unicode normalization, and regex rules.

**Expected action output:** OOV dashboard with source breakdown (language/domain/channel) and rollback decision if threshold breached.

### Q16) p95 latency regresses after tokenizer change. What rollback and mitigation steps do you take?

1. **Immediate safety**
- Trigger canary rollback to last stable tokenizer artifact.
- Freeze rollout and preserve logs for diff analysis.

2. **Root-cause isolation**
- Compare p50/p95 latency, CPU, memory, and sequence-length distribution before/after change.
- Check if token count inflation caused longer downstream inference time.

3. **Mitigation path**
- Optimize with Fast tokenizer variants, trim strategy tuning, and batch/pipeline improvements.
- Re-release via staged canary with rollback guardrails.

**Expected action output:** incident report + corrected release plan + updated latency SLO gates.

### Q17) Retrieval quality drops after index refresh. How do you verify tokenizer parity between query and index?

1. Re-tokenize a golden query set and sampled index docs with both old/new pipelines.
2. Compare token IDs, normalization outputs, and OOV rates side-by-side.
3. Validate chunking policy and boundary overlap consistency.
4. Recompute retrieval metrics (Recall@k, MRR, NDCG) on a fixed benchmark.

**Expected action output:** parity diff report proving whether mismatch is tokenizer, chunking, or embedding-side.

### Q18) NER spans drift after normalization update. What validation suite do you run?

1. **Offset integrity tests**
- Validate normalized-to-original offset mapping on multilingual and accented samples.

2. **Span accuracy tests**
- Run entity span F1 and exact-match checks on a labeled regression suite.

3. **Constraint and decode checks**
- Verify tag-sequence validity and constrained decoding behavior under new normalization.

4. **Edge-case suite**
- Include punctuation-heavy, emoji, mixed-script, and OCR-noisy inputs.

**Expected action output:** go/no-go based on span F1 floor + offset error threshold.

### Q19) Need to preserve question tokens under tight budget. Which trimmer and KPI prove success?

- Prefer **WaterfallTrimmer** when question segment has strict business priority.
- Alternative: **RoundRobinTrimmer** when fairness across question/context is required.

**Primary KPI:** question-preservation ratio.

**Supporting KPIs:**
- answer quality (EM/F1)
- retrieval quality (Recall@k)
- latency/throughput impact after trimming

**Expected action output:** selected trimmer with KPI evidence and acceptance threshold.

### Q20) Which release strategy best reduces tokenizer risk in enterprise?

1. Artifact version pinning (tokenizer, vocab, model, preprocessing config).
2. Progressive delivery (shadow -> canary -> phased rollout).
3. Automatic rollback on KPI/SLO breach (latency, OOV drift, quality).
4. Continuous parity tests in CI/CD for train/serve preprocessing.
5. Post-release monitoring with alerting and weekly drift review.

**Expected action output:** standardized release playbook with rollback SLA and ownership matrix.

## 1G) Enterprise Case-Study Drill Sheet (Organized by Use Case)

### Case 1: Customer Support Routing (High Throughput API)

- Problem: misrouting due to noisy user text and abbreviations.
- Recommended stack: `case_fold_utf8` + subword tokenizer + stable vocab table.
- Success KPI: routing F1, p95 latency, OOV rate.
- Failure signal: high OOV and class confusion on noisy inputs.

### Case 2: KYC NER (Compliance Critical)

- Problem: entity spans incorrect after normalization updates.
- Recommended stack: normalization with offset mapping + constrained decoding.
- Success KPI: entity span F1, offset consistency error rate.
- Failure signal: detected entities cannot be mapped to original text slices.

### Case 3: Retrieval for Policy Search

- Problem: index/query mismatch after tokenizer upgrade.
- Recommended stack: shared normalization and tokenizer config for both paths.
- Success KPI: recall@k, MRR/NDCG, query OOV drift.
- Failure signal: retrieval drop only on newly indexed documents.

### Case 4: Long-Document Compliance QA

- Problem: answer spans lost near chunk boundaries.
- Recommended stack: `sliding_window` with overlap + chunk merge rules.
- Success KPI: chunk coverage rate, boundary miss rate, answer EM/F1.
- Failure signal: frequent misses at chunk edges.

### Case 5: Mobile On-Device Inference

- Problem: latency and memory exceed SLO.
- Recommended stack: Fast tokenizer variants + compact model packaging.
- Success KPI: on-device p95 latency, memory footprint, battery-impact proxy.
- Failure signal: unstable response time across device tiers.

### Standard Enterprise Exit Checklist

- Tokenizer/vocab/model version lock is enforced.
- Train/serve parity tests are automated.
- Canary + rollback plan exists for tokenizer changes.
- KPI dashboard includes latency, throughput, retention, OOV drift, quality.
- Compliance tasks include offset and span integrity tests.

## 2) TensorFlow Text Practical Learning Path (Basic -> Advanced)

This section is focused on learning `tensorflow_text` with practical ML concepts, certification-style thinking, and interview-ready understanding.

### Learning objectives

- Understand how text moves from raw strings to model-ready tensors.
- Map each preprocessing step to an ML concept (normalization, tokenization, trimming, padding, masking).
- Practice choosing APIs by use case (classification, NER, retrieval, long-context QA).
- Build intuition for enterprise KPIs (latency, throughput, OOV rate, retention ratio).

### Structure

1. **Cell A (Basic):** Normalize + tokenize + vocab lookup.
2. **Cell B (Intermediate):** Pair input packing for transformer models.
3. **Cell C (Advanced):** Long-document chunking + trimmer strategy + KPI.
4. **Cell D (Certification/Interview):** Decision drill with explanation-oriented outputs.

In [4]:
# Cell A (Basic): Normalize -> Tokenize -> Vocabulary Lookup
import tensorflow as tf
import tensorflow_text as tf_text

basic_texts = tf.constant([
    'Café refund request',
    'TRACKING update needed',
    'Account issue: OTP failed',
])

# ML concept 1: normalization improves consistency and vocab hit rate
basic_norm = tf_text.case_fold_utf8(basic_texts)

# ML concept 2: tokenization turns text into model-consumable units
basic_tok = tf_text.WhitespaceTokenizer()
basic_tokens = basic_tok.tokenize(basic_norm)

# ML concept 3: map tokens to IDs for models
basic_vocab = [
    '[PAD]', '[UNK]', 'cafe', 'refund', 'request', 'tracking',
    'update', 'needed', 'account', 'issue:', 'otp', 'failed',
]
basic_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=basic_vocab,
        values=tf.cast(tf.range(len(basic_vocab)), tf.int64),
    ),
    num_oov_buckets=1,
)
basic_ids = basic_table.lookup(basic_tokens)

print('=== BASIC PIPELINE ===')
for raw, norm, toks, ids in zip(
    basic_texts.numpy(), basic_norm.numpy(), basic_tokens.to_list(), basic_ids.to_list()
):
    print(f"raw={raw.decode('utf-8')!r}")
    print(f"norm={norm.decode('utf-8')!r}")
    print('tokens=', [t.decode('utf-8') for t in toks])
    print('ids   =', ids)
    print('-' * 60)

print('Interview tip: explain why normalization before tokenization improves OOV behavior.')

=== BASIC PIPELINE ===
raw='Café refund request'
norm='café refund request'
tokens= ['café', 'refund', 'request']
ids   = [12, 3, 4]
------------------------------------------------------------
raw='TRACKING update needed'
norm='tracking update needed'
tokens= ['tracking', 'update', 'needed']
ids   = [5, 6, 7]
------------------------------------------------------------
raw='Account issue: OTP failed'
norm='account issue: otp failed'
tokens= ['account', 'issue:', 'otp', 'failed']
ids   = [8, 9, 10, 11]
------------------------------------------------------------
Interview tip: explain why normalization before tokenization improves OOV behavior.


In [5]:
# Cell B (Intermediate): Pair Input Packing for Transformer Models
# Use case: question-answer pair classification under token budget

q_texts = tf.constant([
    'where is my refund status',
    'how to reset account password',
])
c_texts = tf.constant([
    'refund status is visible in billing dashboard after settlement',
    'use forgot password flow and verify otp to reset credentials',
])

pair_tok = tf_text.WhitespaceTokenizer()
q_toks = pair_tok.tokenize(tf_text.case_fold_utf8(q_texts))
c_toks = pair_tok.tokenize(tf_text.case_fold_utf8(c_texts))

max_content_len = 12
pair_trimmer = tf_text.WaterfallTrimmer(max_seq_length=max_content_len, axis=1)
q_trim, c_trim = pair_trimmer.trim([q_toks, c_toks])

# Convert to simple IDs using shared vocab table
pair_vocab = [
    '[PAD]', '[UNK]', '[CLS]', '[SEP]', 'where', 'is', 'my', 'refund', 'status',
    'how', 'to', 'reset', 'account', 'password', 'visible', 'in', 'billing',
    'dashboard', 'after', 'settlement', 'use', 'forgot', 'flow', 'and', 'verify',
    'otp', 'credentials',
]
pair_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=pair_vocab,
        values=tf.cast(tf.range(len(pair_vocab)), tf.int64),
    ),
    num_oov_buckets=1,
)

q_ids = pair_table.lookup(q_trim)
c_ids = pair_table.lookup(c_trim)

cls_id = tf.constant([[2]], dtype=tf.int64)
sep_id = tf.constant([[3]], dtype=tf.int64)

packed = []
type_ids = []
for qi, ci in zip(q_ids.to_list(), c_ids.to_list()):
    row = [2] + qi + [3] + ci + [3]
    trow = [0] * (len(qi) + 2) + [1] * (len(ci) + 1)
    packed.append(row)
    type_ids.append(trow)

packed_rt = tf.ragged.constant(packed, dtype=tf.int64)
types_rt = tf.ragged.constant(type_ids, dtype=tf.int64)
input_ids, attn_mask = tf_text.pad_model_inputs(packed_rt, max_seq_length=20)
token_types, _ = tf_text.pad_model_inputs(types_rt, max_seq_length=20)

print('=== INTERMEDIATE PIPELINE (PAIR PACKING) ===')
print('input_ids:')
print(input_ids.numpy().tolist())
print('attention_mask:')
print(attn_mask.numpy().tolist())
print('token_type_ids:')
print(token_types.numpy().tolist())
print('Certification tip: choose Waterfall when query/question preservation has business priority.')

=== INTERMEDIATE PIPELINE (PAIR PACKING) ===
input_ids:
[[2, 4, 5, 6, 7, 8, 3, 7, 8, 5, 14, 15, 16, 17, 3, 0, 0, 0, 0, 0], [2, 9, 10, 11, 12, 13, 3, 20, 21, 13, 22, 23, 24, 25, 3, 0, 0, 0, 0, 0]]
attention_mask:
[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]
token_type_ids:
[[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]
Certification tip: choose Waterfall when query/question preservation has business priority.


In [6]:
# Cell C (Advanced): Long-Document Chunking + KPI Snapshot
# Use case: compliance QA over long documents

long_docs = tf.constant([
    'policy update requires identity verification before refund processing and audit logging for every transaction event',
    'security guideline mandates otp validation device binding anomaly checks and escalation workflow for suspicious access',
])

adv_tok = tf_text.WhitespaceTokenizer()
adv_tokens = adv_tok.tokenize(tf_text.case_fold_utf8(long_docs))

window = 8
stride = 5

def build_overlapping_chunks(token_rows, width, step):
    rows = []
    for row in token_rows.to_list():
        starts = list(range(0, max(len(row) - width + 1, 1), step))
        if len(row) > width and (len(row) - width) not in starts:
            starts.append(len(row) - width)
        chunks = [row[s:s + width] for s in starts]
        rows.append(chunks)
    return rows

chunk_rows = build_overlapping_chunks(adv_tokens, window, stride)

# KPI-oriented summary
total_tokens = [len(r) for r in adv_tokens.to_list()]
chunk_counts = [len(c) for c in chunk_rows]
coverage_ratio = []
for row, chunks in zip(adv_tokens.to_list(), chunk_rows):
    covered = set()
    for ch in chunks:
        for tok in ch:
            covered.add(tok)
    coverage_ratio.append(len(covered) / max(len(row), 1))

print('=== ADVANCED PIPELINE (LONG DOC CHUNKING) ===')
for i, (tt, cc, cr) in enumerate(zip(total_tokens, chunk_counts, coverage_ratio), start=1):
    print(f'doc_{i}: total_tokens={tt}, chunk_count={cc}, coverage_ratio={cr:.3f}')

print('KPI note: monitor chunk_count, coverage_ratio, and downstream answer-quality metrics together.')
print('Interview tip: overlap reduces boundary misses in long-context QA.')

=== ADVANCED PIPELINE (LONG DOC CHUNKING) ===
doc_1: total_tokens=15, chunk_count=3, coverage_ratio=1.000
doc_2: total_tokens=15, chunk_count=3, coverage_ratio=1.000
KPI note: monitor chunk_count, coverage_ratio, and downstream answer-quality metrics together.
Interview tip: overlap reduces boundary misses in long-context QA.


In [7]:
# Cell D (Certification + Interview): API Decision Drill
scenarios = [
    {
        'name': 'Multilingual chat moderation',
        'constraint': 'mixed scripts, noisy user text',
        'best_choice': 'UnicodeScriptTokenizer or SentencePiece',
        'why': 'better script boundaries and robust subword coverage',
    },
    {
        'name': 'Question-first pair classification',
        'constraint': 'strict token budget, preserve question tokens',
        'best_choice': 'WaterfallTrimmer',
        'why': 'keeps earlier segment priority under budget',
    },
    {
        'name': 'Long-document compliance QA',
        'constraint': 'context longer than model window',
        'best_choice': 'sliding_window with overlap',
        'why': 'reduces boundary information loss',
    },
    {
        'name': 'Mobile on-device inference',
        'constraint': 'low latency + portable deployment',
        'best_choice': 'Fast tokenizer variants',
        'why': 'better deployment profile and runtime efficiency',
    },
]

print('=== CERTIFICATION + INTERVIEW DECISION DRILL ===')
for i, s in enumerate(scenarios, start=1):
    print(f"\n{i}. {s['name']}")
    print(f"   Constraint : {s['constraint']}")
    print(f"   Best API   : {s['best_choice']}")
    print(f"   Why        : {s['why']}")

print('\nHow to answer in interview:')
print('- State task objective')
print('- State constraint (quality/latency/context)')
print('- Choose API and justify trade-off')
print('- Mention one KPI to validate the decision')

=== CERTIFICATION + INTERVIEW DECISION DRILL ===

1. Multilingual chat moderation
   Constraint : mixed scripts, noisy user text
   Best API   : UnicodeScriptTokenizer or SentencePiece
   Why        : better script boundaries and robust subword coverage

2. Question-first pair classification
   Constraint : strict token budget, preserve question tokens
   Best API   : WaterfallTrimmer
   Why        : keeps earlier segment priority under budget

3. Long-document compliance QA
   Constraint : context longer than model window
   Best API   : sliding_window with overlap
   Why        : reduces boundary information loss

4. Mobile on-device inference
   Constraint : low latency + portable deployment
   Best API   : Fast tokenizer variants
   Why        : better deployment profile and runtime efficiency

How to answer in interview:
- State task objective
- State constraint (quality/latency/context)
- Choose API and justify trade-off
- Mention one KPI to validate the decision


In [8]:
# text.WhitespaceTokenizer
whitespace_tokenizer = tf_text.WhitespaceTokenizer()

whitespace_samples = tf.constant([
    "TensorFlow Text is fun",                 # common
    "  extra   spaces\tand tabs  ",           # repeated spaces + tabs
    "",                                        # empty string edge case
    "punctuation,stays?attached!"              # punctuation behavior
])

whitespace_tokens = whitespace_tokenizer.tokenize(whitespace_samples)

for s, row in zip(whitespace_samples.numpy(), whitespace_tokens.to_list()):
    print(s.decode("utf-8"), "->", [t.decode("utf-8") for t in row])

TensorFlow Text is fun -> ['TensorFlow', 'Text', 'is', 'fun']
  extra   spaces	and tabs   -> ['extra', 'spaces', 'and', 'tabs']
 -> []
punctuation,stays?attached! -> ['punctuation,stays?attached!']


In [9]:
# text.UnicodeScriptTokenizer
unicode_tokenizer = tf_text.UnicodeScriptTokenizer()

unicode_samples = tf.constant([
    "hello世界",                 # mixed Latin + CJK
    "email:test@example.com",    # punctuation-heavy
    "",                           # empty string edge case
    "مرحبا123"                    # Arabic + digits
])

unicode_tokens = unicode_tokenizer.tokenize(unicode_samples)

for s, row in zip(unicode_samples.numpy(), unicode_tokens.to_list()):
    print(s.decode("utf-8"), "->", [t.decode("utf-8") for t in row])

hello世界 -> ['hello', '世界']
email:test@example.com -> ['email', ':', 'test', '@', 'example', '.', 'com']
 -> []
مرحبا123 -> ['مرحبا', '123']


In [10]:
# text.PhraseTokenizer
phrases = [
    "<UNK>",
    "machine learning",
    "TensorFlow",
    "New York",
    "ice cream",
]

phrase_tokenizer = tf_text.PhraseTokenizer(
    vocab=phrases,
    unknown_token="<UNK>",
    token_out_type=tf.string,
    support_detokenization=False,
)

phrase_samples = tf.constant([
    "I like machine learning with TensorFlow",  # matches two phrases
    "I visited New York for ice cream",         # matches two different phrases
    "",                                          # empty string edge case
    "no configured phrase here"                  # no-match edge case
])

phrase_tokens = phrase_tokenizer.tokenize(phrase_samples)

for s, row in zip(phrase_samples.numpy(), phrase_tokens.to_list()):
    print(s.decode("utf-8"), "->", [t.decode("utf-8") for t in row])

I like machine learning with TensorFlow -> ['<UNK>', '<UNK>', 'machine learning', '<UNK>', 'TensorFlow']
I visited New York for ice cream -> ['<UNK>', '<UNK>', 'New York', '<UNK>', 'ice cream']
 -> []
no configured phrase here -> ['<UNK>', '<UNK>', '<UNK>', '<UNK>']


In [11]:
# text.ByteSplitter
byte_splitter = tf_text.ByteSplitter()

byte_samples = tf.constant([
    "abc",          # common ASCII
    "café",         # multi-byte UTF-8 character
    "",             # empty string edge case
    "🙂"            # emoji edge case
])

byte_tokens = byte_splitter.split(byte_samples)

for s, row in zip(byte_samples.numpy(), byte_tokens.to_list()):
    print(s.decode("utf-8"), "->", row)

abc -> [97, 98, 99]
café -> [99, 97, 102, 195, 169]
 -> []
🙂 -> [240, 159, 153, 130]


In [12]:
# text.StateBasedSentenceBreaker
sentence_breaker = tf_text.StateBasedSentenceBreaker()

sentence_samples = tf.constant([
    "Hello world. TensorFlow Text is useful!",      # normal punctuation
    "No punctuation but still one sentence",         # edge: no sentence end mark
    "",                                               # empty string edge case
    "Mr. Smith went home. Dr. Lee stayed."           # abbreviation edge case
])

broken = sentence_breaker.break_sentences(sentence_samples)

for s, row in zip(sentence_samples.numpy(), broken.to_list()):
    print(s.decode("utf-8"), "->", [t.decode("utf-8") for t in row])

Hello world. TensorFlow Text is useful! -> ['Hello world.', 'TensorFlow Text is useful!']
No punctuation but still one sentence -> ['No punctuation but still one sentence']
 -> []
Mr. Smith went home. Dr. Lee stayed. -> ['Mr.', 'Smith went home.', 'Dr.', 'Lee stayed.']


In [13]:
# text.BertTokenizer
bert_vocab = tf.constant([
    "[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]",
    "hello", "world", "tensor", "##flow", "un", "##seen", "!"
])

bert_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=bert_vocab,
        values=tf.range(tf.size(bert_vocab, out_type=tf.int64), dtype=tf.int64),
    ),
    num_oov_buckets=1,
    lookup_key_dtype=tf.string,
    name="bert_vocab_table",
)

bert_tokenizer = tf_text.BertTokenizer(bert_table, lower_case=True)

bert_samples = tf.constant([
    "Hello world!",        # common
    "TensorFlow",          # should split to tensor + ##flow if in vocab
    "",                    # empty string edge case
    "unseen_token"         # unknown piece edge case
])

bert_ids = bert_tokenizer.tokenize(bert_samples)

for s, row in zip(bert_samples.numpy(), bert_ids.to_list()):
    print(s.decode("utf-8"), "->", row)

Hello world! -> [[5], [6], [11]]
TensorFlow -> [[7, 8]]
 -> []
unseen_token -> [[9, 10], [1], [1]]


In [14]:
# text.FastBertTokenizer
fast_bert_vocab = [
    "[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]",
    "hello", "world", "tensor", "##flow", "un", "##seen", "!"
]

fast_bert_samples = tf.constant([
    "Hello world!",
    "TensorFlow",
    "",
    "unseen_token",
])

try:
    fast_bert = tf_text.FastBertTokenizer(fast_bert_vocab, token_out_type=tf.int64)
    fast_bert_ids = fast_bert.tokenize(fast_bert_samples)
    for s, row in zip(fast_bert_samples.numpy(), fast_bert_ids.to_list()):
        print(s.decode("utf-8"), "->", row)
except Exception as e:
    print("FastBertTokenizer setup can vary by tensorflow_text version.")
    print("Error:", e)

Hello world! -> [1, 6, 11]
TensorFlow -> [1]
 -> []
unseen_token -> [9, 10, 1, 1]


In [15]:
# text.FastWordpieceTokenizer
fast_wp_vocab = [
    "[PAD]", "[UNK]", "hello", "world", "tensor", "##flow", "un", "##seen", "!"
]

fast_wp_samples = tf.constant([
    "hello world",        # common
    "TensorFlow",         # composed token
    "",                   # empty edge case
    "unseen_token !",     # unknown + punctuation edge case
])

try:
    fast_wp = tf_text.FastWordpieceTokenizer(
        vocab=fast_wp_vocab,
        unknown_token="[UNK]",
        token_out_type=tf.int64,
    )
    fast_wp_ids = fast_wp.tokenize(fast_wp_samples)
    for s, row in zip(fast_wp_samples.numpy(), fast_wp_ids.to_list()):
        print(s.decode("utf-8"), "->", row)
except Exception as e:
    print("FastWordpieceTokenizer setup can vary by tensorflow_text version.")
    print("Error:", e)

hello world -> [2, 3]
TensorFlow -> [1]
 -> []
unseen_token ! -> [6, 7, 1, 1, 8]


In [16]:
# text.FastSentencepieceTokenizer
import os
import tempfile
import importlib.util
import subprocess
import sys

# Create a tiny local SentencePiece model if it does not exist.
sentencepiece_model_path = "tiny_sentencepiece.model"

sp_samples = tf.constant([
    "hello world",
    "",
    "TensorFlow-text",
    "rareword123",
])

try:
    if importlib.util.find_spec("sentencepiece") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "sentencepiece"])

    import sentencepiece as spm

    if not tf.io.gfile.exists(sentencepiece_model_path):
        corpus_lines = [
            "hello world",
            "hello tensorflow text",
            "machine learning with tensorflow",
            "tokenization with sentencepiece",
            "rareword123 and symbols like hyphen-text",
        ]
        corpus_path = os.path.join(tempfile.gettempdir(), "tiny_sp_corpus.txt")
        with tf.io.gfile.GFile(corpus_path, "w") as f:
            for line in corpus_lines:
                f.write(line + "\n")

        spm.SentencePieceTrainer.Train(
            input=corpus_path,
            model_prefix="tiny_sentencepiece",
            vocab_size=32,
            model_type="unigram",
            character_coverage=1.0,
            hard_vocab_limit=False,
            bos_id=-1,
            eos_id=-1,
            pad_id=-1,
        )

    model_bytes = tf.io.gfile.GFile(sentencepiece_model_path, "rb").read()
    fast_sp = tf_text.FastSentencepieceTokenizer(model=model_bytes)
    sp_ids = fast_sp.tokenize(sp_samples)

    for s, row in zip(sp_samples.numpy(), sp_ids.to_list()):
        print(s.decode("utf-8"), "->", row)

except Exception as e:
    print("Could not train/load SentencePiece model automatically.")
    print("Error:", e)

hello world -> [12, 1, 5, 5, 9, 13, 9, 21, 5, 11]
 -> []
TensorFlow-text -> [3, 0, 8, 20, 9, 21, 0, 5, 9, 31, 25, 2, 1, 17, 2]
rareword123 -> [3, 21, 10, 21, 1, 31, 9, 21, 11, 26, 27, 28]


---
## 2) Additional Tokenizers in `tensorflow_text`

The following tokenizers extend the core set with full subword, character-level, and model-based tokenization strategies.

| Tokenizer | Category | Key Use |
|---|---|---|
| `SentencepieceTokenizer` | Subword | Full SentencePiece with encode/decode |
| `SplitMergeTokenizer` | Supervised | Split/merge driven by binary label sequence |
| `SplitMergeFromLogitsTokenizer` | Supervised | Split/merge driven by raw logit scores |
| `UnicodeCharTokenizer` | Character | One token per Unicode character |
| `WordpieceTokenizer` | Subword | Classic WordPiece with detokenize support |
| `HubModuleTokenizer` | Hub | Any tokenizer SavedModel from TF Hub |

### `text.SentencepieceTokenizer`

**Purpose:** Full SentencePiece tokenizer — trains/loads a `.model` file. Supports encode/decode, byte-pair, unigram, and char models. More feature-rich than `FastSentencepieceTokenizer`.

| | |
|---|---|
| **Do** | Use when you need `detokenize`, sampling modes, or BPE/char model types |
| **Don't** | Use in production TF graphs for export — use `FastSentencepieceTokenizer` instead |
| **Tip** | `add_bos`/`add_eos` args control start/end tokens; use `out_type=tf.string` for readable output |

In [17]:
# text.SentencepieceTokenizer  (reuses tiny_sentencepiece.model from previous cell)
model_bytes = tf.io.gfile.GFile("tiny_sentencepiece.model", "rb").read()

# --- encode as IDs (default) ---
sp_id_tok = tf_text.SentencepieceTokenizer(model=model_bytes, out_type=tf.int32)

samples = tf.constant(["hello world", "", "TensorFlow-text", "rareword123"])

ids = sp_id_tok.tokenize(samples)
print("IDs:")
for s, row in zip(samples.numpy(), ids.to_list()):
    print(f"  {s.decode():20s} -> {row}")

# --- encode as strings ---
sp_str_tok = tf_text.SentencepieceTokenizer(model=model_bytes, out_type=tf.string)
pieces = sp_str_tok.tokenize(samples)
print("\nPieces:")
for s, row in zip(samples.numpy(), pieces.to_list()):
    print(f"  {s.decode():20s} -> {[p.decode() for p in row]}")

# --- detokenize (round-trip) ---
decoded = sp_id_tok.detokenize(ids)
print("\nRound-trip:")
for orig, rec in zip(samples.numpy(), decoded.numpy()):
    print(f"  original={orig.decode()!r}  recovered={rec.decode()!r}")

IDs:
  hello world          -> [12, 1, 5, 5, 9, 13, 9, 21, 5, 11]
                       -> []
  TensorFlow-text      -> [3, 0, 8, 20, 9, 21, 0, 5, 9, 31, 25, 2, 1, 17, 2]
  rareword123          -> [3, 21, 10, 21, 1, 31, 9, 21, 11, 26, 27, 28]

Pieces:
  hello world          -> ['▁h', 'e', 'l', 'l', 'o', '▁w', 'o', 'r', 'l', 'd']
                       -> []
  TensorFlow-text      -> ['▁', 'T', 'en', 's', 'o', 'r', 'F', 'l', 'o', 'w', '-', 't', 'e', 'x', 't']
  rareword123          -> ['▁', 'r', 'a', 'r', 'e', 'w', 'o', 'r', 'd', '1', '2', '3']

Round-trip:
  original='hello world'  recovered='hello world'
  original=''  recovered=''
  original='TensorFlow-text'  recovered=' ⁇ ensor ⁇ low-text'
  original='rareword123'  recovered='rareword123'


### `text.SplitMergeTokenizer`

**Purpose:** Supervised tokenizer controlled by a **binary label sequence** per character: `0` = merge with previous, `1` = start a new token. Useful when a model predicts token boundaries.

| | |
|---|---|
| **Do** | Use after a sequence labeler that predicts split/merge decisions |
| **Don't** | Confuse label length — it must match the character count of the input string |
| **Tip** | Label `1` at position 0 is ignored; every token implicitly starts at 0 |

In [18]:
# text.SplitMergeTokenizer
# Input : whole strings  (one per batch element)
# Labels: int array with ONE label PER CHARACTER in the string
#   0 = SPLIT  -> start a NEW token at this character
#   1 = MERGE  -> continue / append to the current token
sm_tokenizer = tf_text.SplitMergeTokenizer()

# Common: 'HelloWorld' (10 chars) -> split into ['Hello', 'World']
# H e l l o W o r l d
# 0 1 1 1 1 0 1 1 1 1   (0 = new token, 1 = continue)
inputs = tf.constant(["HelloWorld"])
labels = tf.ragged.constant([[0, 1, 1, 1, 1, 0, 1, 1, 1, 1]])
print("Split Hello|World:", [t.decode() for t in sm_tokenizer.tokenize(inputs, labels).to_list()[0]])

# All-merge -> one token 'abc'
inputs2 = tf.constant(["abc"])
labels2 = tf.ragged.constant([[0, 1, 1]])   # start at 'a', merge 'b' and 'c'
print("All merged        :", [t.decode() for t in sm_tokenizer.tokenize(inputs2, labels2).to_list()[0]])

# Every-char split -> each char is its own token
labels3 = tf.ragged.constant([[0, 0, 0]])   # each char starts a new token
print("All split         :", [t.decode() for t in sm_tokenizer.tokenize(inputs2, labels3).to_list()[0]])

# Batch: two strings with different split patterns
batch_in  = tf.ragged.constant(["Hi", "World"])
batch_lbl = tf.ragged.constant([[0, 1],            # 'Hi'    -> ['Hi']
                                 [0, 1, 0, 1, 1]]) # 'World' -> ['Wo', 'rld']
result    = sm_tokenizer.tokenize(batch_in, batch_lbl).to_list()
print("Batch             :", [[t.decode() for t in row] for row in result])

# Edge: single character
inputs4 = tf.constant(["X"])
labels4 = tf.ragged.constant([[0]])
print("Single char       :", [t.decode() for t in sm_tokenizer.tokenize(inputs4, labels4).to_list()[0]])

Split Hello|World: ['Hello', 'World']
All merged        : ['abc']
All split         : ['a', 'b', 'c']
Batch             : [['Hi'], ['Wo', 'rld']]
Single char       : ['X']


### `text.SplitMergeFromLogitsTokenizer`

**Purpose:** Same as `SplitMergeTokenizer` but accepts **raw logits** (shape `[batch, chars, 2]`) rather than hard binary labels. Class 0 = merge, class 1 = split. Typically used directly after a character-level classifier head.

| | |
|---|---|
| **Do** | Connect directly to the output of a `Dense(2)` layer over character embeddings |
| **Don't** | Apply softmax before passing — the tokenizer picks `argmax` internally |
| **Tip** | Use `force_split_at_break_character=True` to also break at whitespace regardless of logits |

In [19]:
# text.SplitMergeFromLogitsTokenizer
# logits shape: [batch, num_chars, 2]
#   logits[..., 0] = SPLIT score  (high -> start a new token here)
#   logits[..., 1] = MERGE score  (high -> continue current token)
smfl_tokenizer = tf_text.SplitMergeFromLogitsTokenizer()

# String: 'HelloWorld' -> split into ['Hello', 'World']
# Position 0 (H) and 5 (W) get high SPLIT score (col-0)
inputs  = tf.constant(["HelloWorld"])
logits = tf.constant([[
    [9.9, 0.1],   # H -> SPLIT  (start new token)
    [0.1, 9.9],   # e -> merge
    [0.1, 9.9],   # l -> merge
    [0.1, 9.9],   # l -> merge
    [0.1, 9.9],   # o -> merge
    [9.9, 0.1],   # W -> SPLIT  (start new token)
    [0.1, 9.9],   # o -> merge
    [0.1, 9.9],   # r -> merge
    [0.1, 9.9],   # l -> merge
    [0.1, 9.9],   # d -> merge
]])
result = smfl_tokenizer.tokenize(inputs, logits)
print("Logit-driven split:", [t.decode() for t in result.to_list()[0]])

# All high MERGE score -> single token 'abc'
inputs2 = tf.constant(["abc"])
logits2 = tf.constant([[[9.9, 0.1], [0.1, 9.9], [0.1, 9.9]]])  # first=split, rest=merge
print("All merged:        ", [t.decode() for t in smfl_tokenizer.tokenize(inputs2, logits2).to_list()[0]])

# All high SPLIT score -> every char is its own token
logits3 = tf.constant([[[9.9, 0.1], [9.9, 0.1], [9.9, 0.1]]])
print("All split:         ", [t.decode() for t in smfl_tokenizer.tokenize(inputs2, logits3).to_list()[0]])

# Tied logits -> col-0 (SPLIT) wins on tie via argmax
logits4 = tf.constant([[[5.0, 5.0], [5.0, 5.0], [5.0, 5.0]]])
print("Tied (split wins): ", [t.decode() for t in smfl_tokenizer.tokenize(inputs2, logits4).to_list()[0]])

Logit-driven split: ['Hello', 'World']
All merged:         ['abc']
All split:          ['a', 'b', 'c']
Tied (split wins):  ['abc']


### `text.UnicodeCharTokenizer`

**Purpose:** Splits each string into its individual Unicode **codepoints** (one token per character). Works correctly with multi-byte UTF-8 and emoji.

| | |
|---|---|
| **Do** | Use for character-level models (NER, typo correction, char-CNN) |
| **Don't** | Use for large documents — produces very long sequences |
| **Tip** | Returns `int32` codepoints by default; pass `out_type=tf.string` to get UTF-8 chars |

In [20]:
# text.UnicodeCharTokenizer
char_tokenizer = tf_text.UnicodeCharTokenizer()

char_samples = tf.constant([
    "hello",          # common ASCII
    "caf\u00e9",      # multi-byte: é is U+00E9
    "",               # empty edge case
    "\U0001F642",    # emoji: 🙂 (U+1F642)
    "\u4e2d\u6587",  # CJK: 中文
])

# --- as codepoints (int32) ---
codepoints = char_tokenizer.tokenize(char_samples)
print("Codepoints:")
for s, row in zip(char_samples.numpy(), codepoints.to_list()):
    print(f"  {s.decode():12s} -> {row}")

# --- detokenize codepoints back to strings ---
recovered = char_tokenizer.detokenize(codepoints)
print("\nDetokenized:")
for orig, rec in zip(char_samples.numpy(), recovered.numpy()):
    print(f"  {orig.decode()!r:14s} -> {rec.decode()!r}")

Codepoints:
  hello        -> [104, 101, 108, 108, 111]
  café         -> [99, 97, 102, 233]
               -> []
  🙂            -> [128578]
  中文           -> [20013, 25991]

Detokenized:
  'hello'        -> 'hello'
  'café'         -> 'café'
  ''             -> ''
  '🙂'            -> '🙂'
  '中文'           -> '中文'


### `text.WordpieceTokenizer` + Detokenizer

**Purpose:** Classic WordPiece subword tokenizer. Splits words into the longest matching vocabulary subwords, prefixing continuations with `##`. Underlies BERT-family models.

| | |
|---|---|
| **Do** | Use `.detokenize()` to merge `##` pieces back into whole words |
| **Don't** | Pass raw sentences directly — pre-split into words first (e.g. via `WhitespaceTokenizer`) |
| **Tip** | `token_out_type=tf.string` lets you inspect actual subword pieces rather than IDs |

**Detokenizer:** Any tokenizer with a `.detokenize()` method reverses the tokenization. Not all tokenizers support this (e.g. `WhitespaceTokenizer` has no detokenizer).

In [21]:
# text.WordpieceTokenizer + Detokenizer (detokenize shown separately)
wp_vocab_terms = ["[PAD]", "[UNK]", "hello", "world", "tensor", "##flow",
                  "un", "##known", "play", "##ing", "!", "##s"]

wp_lookup = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=wp_vocab_terms,
        values=tf.cast(tf.range(len(wp_vocab_terms)), tf.int64),
    ),
    num_oov_buckets=1,
)

wp_tokenizer = tf_text.WordpieceTokenizer(
    wp_lookup,
    token_out_type=tf.string,   # return pieces as strings
    unknown_token="[UNK]",
    split_unknown_characters=False,
)

# Pre-tokenize into words first (WordpieceTokenizer works word-level)
ws_tok = tf_text.WhitespaceTokenizer()

raw = tf.constant([
    "hello world",        # common: in vocab
    "TensorFlow",         # split: tensor + ##flow
    "unknown playing",    # split: un + ##known, play + ##ing
    "",                   # empty edge case
    "worlds !",           # world + ##s, !
  ])

words        = ws_tok.tokenize(raw)          # shape [batch, words]
wp_pieces    = wp_tokenizer.tokenize(words)  # shape [batch, words, pieces]

print("WordPiece subword tokenization:")
for s, batch_rows in zip(raw.numpy(), wp_pieces.to_list()):
    flat = [p.decode() if isinstance(p, bytes) else p for word_pieces in batch_rows for p in word_pieces]
    print(f"  {s.decode()!r:24s} -> {flat}")

# Note: detokenize() merges ##-prefixed pieces back to words.
# It requires int32 token_ids matching vocab indices exactly (no OOV buckets).
# Usage: wp_tokenizer.detokenize(token_ids_int32) -> word_tensor

WordPiece subword tokenization:
  'hello world'            -> ['hello', 'world']
  'TensorFlow'             -> ['[UNK]']
  'unknown playing'        -> ['un', '##known', 'play', '##ing']
  ''                       -> []
  'worlds !'               -> ['world', '##s', '!']


### `text.HubModuleTokenizer`

**Purpose:** Wraps any **TF Hub SavedModel** that exposes a tokenizer interface. Lets you swap in any pretrained tokenizer (e.g. BERT, mBERT, ALBERT) without writing custom loading code.

| | |
|---|---|
| **Do** | Use for quick experiments with Hub-hosted BERT/ALBERT tokenizers |
| **Don't** | Use in offline environments — requires network access to TF Hub |
| **Tip** | The hub model must export a `tokenize` signature; not all Hub assets do |

> **Note:** This cell is guarded — it requires `tensorflow_hub` and network access. It prints a clear message if either is unavailable.

In [22]:
# text.HubModuleTokenizer  (guarded: requires tensorflow_hub + network access)
import importlib.util

HUB_MODEL = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"

hub_samples = tf.constant(["Hello world!", "TensorFlow text processing."])

if importlib.util.find_spec("tensorflow_hub") is None:
    print("tensorflow_hub not installed. Run: pip install tensorflow-hub")
else:
    try:
        import tensorflow_hub as hub  # noqa: F401
        hub_tokenizer = tf_text.HubModuleTokenizer(HUB_MODEL)

        # tokenize -> returns dict with input_ids, input_mask, segment_ids
        result = hub_tokenizer.tokenize(hub_samples)
        print("Token IDs:", result.to_list())
    except Exception as e:
        print("HubModuleTokenizer requires network access and a compatible Hub model.")
        print("Hub URL used :", HUB_MODEL)
        print("Error        :", e)

tensorflow_hub not installed. Run: pip install tensorflow-hub


---
## 3) Text Processing Utilities

Beyond tokenization, `tensorflow_text` provides utilities for every stage of the NLP pre-processing pipeline.

| Utility | Purpose |
|---|---|
| `ShrinkLongestTrimmer` | Trim by repeatedly removing from the longest sequence |
| `RoundRobinTrimmer` | Trim by taking one token from each sequence in turns |
| `WaterfallTrimmer` | Fill budget greedily: saturate seq-1 first, then seq-2, … |
| `RegexSplitter` | Split on any regex pattern |
| `pad_model_inputs` | Pad ragged tensor → dense + create attention mask |
| `trim_model_inputs` | Trim sequences to a max length |
| `sliding_window` | Build overlapping windows over a sequence |
| `wordshape` | Boolean feature flags per token (ALL_CAPS, HAS_DIGIT, …) |
| `normalize_utf8` | Unicode normalization (NFC / NFKC / NFD / NFKD) |
| `case_fold_utf8` | Lowercase + unicode normalization in one call |

### Trimmers: `ShrinkLongestTrimmer` · `RoundRobinTrimmer` · `WaterfallTrimmer`

**Purpose:** Trimmers reduce multi-segment inputs to fit within a maximum sequence length budget (e.g. 512 for BERT). Each strategy decides which segment to trim next.

| Strategy | Removes from | Best for |
|---|---|---|
| `ShrinkLongest` | Longest remaining segment | Balanced document pairs |
| `RoundRobin` | Each segment in turn | Equal-length segments |
| `Waterfall` | Fills quota in order | Prioritise first segment |

| | |
|---|---|
| **Do** | Call `.trim()` before padding; trimmers produce ragged tensors ready for `pad_model_inputs` |
| **Don't** | Use on already-padded dense tensors — work on ragged token ID tensors |
| **Tip** | `generate_mask=True` in `pad_model_inputs` creates the attention mask automatically |

In [23]:
# Trimmers: ShrinkLongestTrimmer | RoundRobinTrimmer | WaterfallTrimmer
# Scenario: two sequences A and B that together exceed a max_length budget

seq_a = tf.ragged.constant([[1, 2, 3, 4, 5, 6, 7, 8]])       # 8 tokens
seq_b = tf.ragged.constant([[10, 20, 30, 40, 50, 60]])         # 6 tokens
max_len = 10   # budget across both sequences

# --- ShrinkLongestTrimmer: removes from whichever is longer ---
shrink = tf_text.ShrinkLongestTrimmer(max_seq_length=max_len, axis=1)
a_s, b_s = shrink.trim([seq_a, seq_b])
print("ShrinkLongest -> A:", a_s.to_list(), "  B:", b_s.to_list())
print(f"  total tokens: {len(a_s[0].numpy()) + len(b_s[0].numpy())} (budget={max_len})")

# --- RoundRobinTrimmer: alternates removal between sequences ---
rr = tf_text.RoundRobinTrimmer(max_seq_length=max_len, axis=1)
a_r, b_r = rr.trim([seq_a, seq_b])
print("\nRoundRobin    -> A:", a_r.to_list(), "  B:", b_r.to_list())
print(f"  total tokens: {len(a_r[0].numpy()) + len(b_r[0].numpy())} (budget={max_len})")

# --- WaterfallTrimmer: fills budget greedily (seq_a first, then seq_b) ---
wf = tf_text.WaterfallTrimmer(max_seq_length=max_len, axis=1)
a_w, b_w = wf.trim([seq_a, seq_b])
print("\nWaterfall     -> A:", a_w.to_list(), "  B:", b_w.to_list())
print(f"  total tokens: {len(a_w[0].numpy()) + len(b_w[0].numpy())} (budget={max_len})")

# --- Edge: sequences already within budget (no trimming needed) ---
short_a = tf.ragged.constant([[1, 2]])
short_b = tf.ragged.constant([[10]])
a_no, b_no = shrink.trim([short_a, short_b])
print("\nNo trim needed -> A:", a_no.to_list(), " B:", b_no.to_list())

ShrinkLongest -> A: [[1, 2, 3, 4, 5]]   B: [[10, 20, 30, 40, 50]]
  total tokens: 10 (budget=10)

RoundRobin    -> A: [[1, 2, 3, 4, 5]]   B: [[10, 20, 30, 40, 50]]
  total tokens: 10 (budget=10)

Waterfall     -> A: [[1, 2, 3, 4, 5, 6, 7, 8]]   B: [[10, 20]]
  total tokens: 10 (budget=10)

No trim needed -> A: [[1, 2]]  B: [[10]]


### `text.RegexSplitter` · `pad_model_inputs` · `trim_model_inputs`

**RegexSplitter** — Splits text at any regex pattern (keeps the split token as a separate unit).

**`pad_model_inputs(tensor, max_seq_length)`** — Converts a ragged tensor to a dense padded tensor and generates an attention mask in one call. The mask is `1` for real tokens, `0` for padding.

**`trim_model_inputs(tensor, max_seq_length)`** — Truncates each row to `max_seq_length`. Use before padding to ensure you never exceed a model's positional embedding limit.

| | |
|---|---|
| **Do** | Always `trim` before `pad` so padding never adds beyond budget |
| **Don't** | Rely on padding alone to handle long sequences — content will be cut silently at the model |
| **Tip** | The attention mask from `pad_model_inputs` is directly usable as `attention_mask` in Hugging Face / Keras BERT layers |

In [24]:
# Text utilities: RegexSplitter, pad_model_inputs

# --- RegexSplitter ---
# Split by whitespace regex
rs = tf_text.RegexSplitter(r'\s+')
texts = tf.constant([
    "split on whitespace",
    "  leading and  trailing  ",
    "",
    "no-space-at-all"
  ])
splits = rs.split(texts)

print("RegexSplitter (\\s+):")
for t, s in zip(texts.numpy(), splits.to_list()):
    print(f"  {t.decode()!r:30s} -> {[x.decode() for x in s]}")

# --- pad_model_inputs ---
# Pad ragged tensor to max_seq_length, return (padded, mask)
def pad_model_inputs(token_ids, max_seq_length):
    """Pad ragged tensor to max_seq_length. Returns (padded_tensor, attention_mask)."""
    padded = token_ids.to_tensor(shape=[None, max_seq_length], default_value=0)
    mask = tf.cast(tf.not_equal(padded, 0), tf.int32)
    return padded, mask

ragged_ids = tf.ragged.constant([[1, 2, 3], [4, 5], [6, 7, 8, 9, 10]])
padded, mask = pad_model_inputs(ragged_ids, max_seq_length=5)

print("\npad_model_inputs (max_seq_length=5):")
print("  padded:", padded.numpy().tolist())
print("  mask:  ", mask.numpy().tolist())

RegexSplitter (\s+):
  'split on whitespace'          -> ['split', 'on', 'whitespace']
  '  leading and  trailing  '    -> ['leading', 'and', 'trailing']
  ''                             -> []
  'no-space-at-all'              -> ['no-space-at-all']

pad_model_inputs (max_seq_length=5):
  padded: [[1, 2, 3, 0, 0], [4, 5, 0, 0, 0], [6, 7, 8, 9, 10]]
  mask:   [[1, 1, 1, 0, 0], [1, 1, 0, 0, 0], [1, 1, 1, 1, 1]]


### `text.sliding_window` · `text.wordshape` · `text.normalize_utf8` · `text.case_fold_utf8`

**`sliding_window(data, width, axis)`** — Produces overlapping windows of fixed width over a sequence. Used for chunking long documents into overlapping model-sized pieces.

**`wordshape(tokens, pattern)`** — Returns boolean flags for orthographic features of each token (ALL_CAPS, HAS_DIGIT, STARTS_WITH_UPPERCASE, IS_PUNCTUATION, etc.).

**`normalize_utf8(text, normalization_form)`** — Applies Unicode normalization (NFC, NFKC, NFD, NFKD). Use to canonicalize text before tokenization.

**`case_fold_utf8(text)`** — Lowercases and applies NFKC normalization in one pass. The same operation BERT's vocab was built on.

| | |
|---|---|
| **Do** | Call `normalize_utf8` / `case_fold_utf8` **before** tokenization for consistent vocab hits |
| **Don't** | Apply normalization after tokenization — the pieces won't match the vocab anymore |
| **Tip** | `wordshape` is useful as auxiliary features in NER taggers alongside token embeddings |

In [25]:
# Text utilities: sliding_window, normalize_utf8, case_fold

# --- sliding_window ---
# Create sliding window of fixed width on a sequence
seq = tf.range(10, 70, 10)  # [10, 20, 30, 40, 50, 60]
windows = tf_text.sliding_window(seq, width=3, axis=0)
print("sliding_window (width=3):")
print(" ", windows.numpy().tolist())

short_seq = tf.constant([1, 2])
short_windows = tf_text.sliding_window(short_seq, width=3, axis=0)
print("Short seq windows:", short_windows.numpy().tolist())

# --- normalize_utf8 & case_fold ---
# Unicode normalization and case folding
samples = tf.constant([
    "hello",           # plain ASCII
    "HELLO",           # uppercase
    "café",            # accented
    "CAFÉ",            # accented + uppercase
    "ﬁ",               # ligature fi
    "Ⅳ",               # Roman numeral
  ])

normalized = tf_text.normalize_utf8(samples, 'NFC')  # NFC = composed form
folded     = tf_text.case_fold_utf8(samples)        # lowercase + normalize

print("\nnormalize_utf8 (NFC form):")
for s, n in zip(samples.numpy(), normalized.numpy()):
    print(f"  {s.decode()!r:15s} -> {n.decode()!r}")

print("\ncase_fold_utf8 (lowercase + normalize):")
for s, f in zip(samples.numpy(), folded.numpy()):
    print(f"  {s.decode()!r:15s} -> {f.decode()!r}")

sliding_window (width=3):
  [[10, 20, 30], [20, 30, 40], [30, 40, 50], [40, 50, 60]]
Short seq windows: []

normalize_utf8 (NFC form):
  'hello'         -> 'hello'
  'HELLO'         -> 'HELLO'
  'café'          -> 'café'
  'CAFÉ'          -> 'CAFÉ'
  'ﬁ'             -> 'ﬁ'
  'Ⅳ'             -> 'Ⅳ'

case_fold_utf8 (lowercase + normalize):
  'hello'         -> 'hello'
  'HELLO'         -> 'hello'
  'café'          -> 'café'
  'CAFÉ'          -> 'café'
  'ﬁ'             -> 'fi'
  'Ⅳ'             -> 'iv'


---
## 4) Grouped Examples

The following cells demonstrate realistic NLP pre-processing **pipelines** that combine multiple `tensorflow_text` APIs together.

| Group | What it shows |
|---|---|
| **Basic pipeline** | Tokenize → pad → attention mask |
| **With Splitter** | RegexSplitter → normalize → IDs |
| **Trimmer** | Trim two-segment input to max length |
| **Reduction / Chunking** | Sliding windows over a long document |
| **Selector / Chooser** | wordshape-driven token filtering |

### Group 1 — Basic Pipeline: Tokenize → Pad → Attention Mask

A complete minimal pipeline: raw strings → WhitespaceTokenizer → vocab lookup → trim → pad → mask.

In [26]:
# ========== EXAMPLE 1: BASIC TOKENIZATION PIPELINE ==========
# Simple end-to-end tokenization with whitespace splitter + vocabulary lookup

# --- Setup constants ---
MAX_LEN = 5
raw_texts = tf.constant([
    "hello world",
    "machine learning is fun",
    "tensorflow text",
])

# --- STEP 1: Pre-process (normalize, lowercase) ---
normalized = tf_text.case_fold_utf8(raw_texts)

# --- STEP 2: Tokenize into words (whitespace) ---
ws_tokenizer = tf_text.WhitespaceTokenizer()
tokens = ws_tokenizer.tokenize(normalized)  # ragged [batch, words]

# --- STEP 3: Vocabulary lookup ---
vocab_terms = ["[PAD]", "[UNK]", "hello", "world", "machine", "learning", "is", "fun", "tensorflow", "text"]
vocab_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=vocab_terms,
        values=tf.cast(tf.range(len(vocab_terms)), tf.int64),
    ),
    num_oov_buckets=1,
)
ids = vocab_table.lookup(tokens)  # ragged [batch, ids]

# --- STEP 4: Pad to max_seq_length + get attention mask ---
def pad_for_model(ids, max_seq_len):
    """Pad ragged to dense and create attention mask."""
    ids_padded = ids.to_tensor(shape=[None, max_seq_len], default_value=0)
    mask = tf.cast(tf.not_equal(ids_padded, 0), tf.int32)
    return ids_padded, mask

ids_padded, attn_mask = pad_for_model(ids, MAX_LEN)

print("Example 1: Basic Pipeline")
print("Input texts:", raw_texts.numpy().tolist())
print("Token IDs:\n", ids_padded.numpy().tolist())
print("Attention mask:\n", attn_mask.numpy().tolist())
print(f"  Shape: {ids_padded.shape} (batch={ids_padded.shape[0]}, max_len={MAX_LEN})")

Example 1: Basic Pipeline
Input texts: [b'hello world', b'machine learning is fun', b'tensorflow text']
Token IDs:
 [[2, 3, 0, 0, 0], [4, 5, 6, 7, 0], [8, 9, 0, 0, 0]]
Attention mask:
 [[1, 1, 0, 0, 0], [1, 1, 1, 1, 0], [1, 1, 0, 0, 0]]
  Shape: (3, 5) (batch=3, max_len=5)


### Group 2 — With Splitter: RegexSplitter → Normalize → Lookup

Demonstrates using `RegexSplitter` on punctuation, then normalizing and looking up IDs — useful for tokenizing structured text like code or CSV.

In [27]:
# Group 2: RegexSplitter -> normalize -> vocab lookup
# Use-case: tokenize comma-separated values or punctuation-heavy text

csv_vocab = ["[PAD]", "[UNK]", "apple", "banana", "cherry", "mango", "1", "2", "3"]
csv_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=csv_vocab,
        values=tf.cast(tf.range(len(csv_vocab)), tf.int64),
    ),
    num_oov_buckets=1,
)

csv_splitter = tf_text.RegexSplitter(split_regex=r"[,\s]+")  # split on comma or whitespace

raw_csv = tf.constant([
    "apple,banana,cherry",      # common CSV
    "mango, 1, 2, 3",           # spaces around commas
    "",                          # empty edge case
    "pear,MANGO",               # mixed case + unknown token
])

# Step 1: normalize (case-fold) so 'MANGO' hits the vocab
normalized = tf_text.case_fold_utf8(raw_csv)

# Step 2: split on commas/spaces
pieces = csv_splitter.split(normalized)   # ragged

# Step 3: vocab lookup
piece_ids = csv_table.lookup(pieces)

print(f"{'Input':30s}  Pieces                         IDs")
for s, tok_row, id_row in zip(raw_csv.numpy(), pieces.to_list(), piece_ids.to_list()):
    toks = [t.decode() for t in tok_row]
    print(f"  {s.decode()!r:28s}  {str(toks):32s}  {id_row}")

Input                           Pieces                         IDs
  'apple,banana,cherry'         ['apple', 'banana', 'cherry']     [2, 3, 4]
  'mango, 1, 2, 3'              ['mango', '1', '2', '3']          [5, 6, 7, 8]
  ''                            []                                []
  'pear,MANGO'                  ['pear', 'mango']                 [9, 5]


### Group 3 — Trimmer: Two-Segment BERT-Style Input

BERT takes a `[CLS] sentence_a [SEP] sentence_b [SEP]` input within 512 tokens. This shows how to prepare and trim two segments using `ShrinkLongestTrimmer`.

In [28]:
# Group 3: Trimmer pipeline for two-segment BERT input
# Budget: 12 tokens including [CLS] + 2x [SEP] = 3 special tokens -> 9 content tokens
CLS, SEP, PAD = 101, 102, 0
MAX_BERT_LEN  = 12
CONTENT_BUDGET = MAX_BERT_LEN - 3   # 9 content tokens

# Simulate two pre-tokenized integer segments
seg_a = tf.ragged.constant([[1, 2, 3, 4, 5, 6, 7]])   # 7 tokens (long)
seg_b = tf.ragged.constant([[10, 20, 30, 40, 50]])    # 5 tokens

# Step 1: trim content to budget
trimmer  = tf_text.ShrinkLongestTrimmer(max_seq_length=CONTENT_BUDGET, axis=1)
ta, tb   = trimmer.trim([seg_a, seg_b])
print(f"After trim -> A: {ta.to_list()}  B: {tb.to_list()}")

# Step 2: add [CLS] / [SEP] special tokens
cls_t = tf.constant([[CLS]])
sep_t = tf.constant([[SEP]])

input_ids = tf.concat([cls_t,
                        ta.to_tensor(),
                        sep_t,
                        tb.to_tensor(),
                        sep_t], axis=1)

# Step 3: build token_type_ids (0 for seg_a, 1 for seg_b)
len_a = ta.row_lengths()[0]
len_b = tb.row_lengths()[0]
type_ids = tf.concat([
    tf.zeros([1, 1 + len_a + 1], dtype=tf.int32),   # [CLS] + seg_a + [SEP]
    tf.ones( [1, len_b + 1],     dtype=tf.int32),   # seg_b + [SEP]
], axis=1)

# Step 4: pad to MAX_BERT_LEN
input_ragged   = tf.RaggedTensor.from_tensor(input_ids)
type_ragged    = tf.RaggedTensor.from_tensor(type_ids)

input_padded, attn_mask = tf_text.pad_model_inputs(input_ragged, max_seq_length=MAX_BERT_LEN)
type_padded,  _         = tf_text.pad_model_inputs(type_ragged,  max_seq_length=MAX_BERT_LEN)

print("\nBERT-style input:")
print("  input_ids    :", input_padded.numpy().tolist()[0])
print("  token_type_ids:", type_padded.numpy().tolist()[0])
print("  attention_mask:", attn_mask.numpy().tolist()[0])

After trim -> A: [[1, 2, 3, 4]]  B: [[10, 20, 30, 40, 50]]

BERT-style input:
  input_ids    : [101, 1, 2, 3, 4, 102, 10, 20, 30, 40, 50, 102]
  token_type_ids: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### Group 4 — Reduction / Chunking: Sliding Windows Over Long Documents

Long documents exceed model context windows. `sliding_window` creates **overlapping chunks** so no content is entirely cut off. A downstream model scores each chunk independently.

In [29]:
# Group 4: Reduction / chunking with sliding_window
WINDOW = 4   # tokens per chunk
STRIDE = 2   # step size (overlap = WINDOW - STRIDE = 2)

# Simulate a long tokenized document (single sequence of IDs)
long_doc_ids = tf.constant([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Create overlapping windows
chunks = tf_text.sliding_window(long_doc_ids, width=WINDOW, axis=0)
print(f"Document length: {len(long_doc_ids.numpy())}  Window={WINDOW}  Stride={STRIDE}")
print("All windows (no stride):", chunks.numpy().tolist())

# Simulate strided chunking via gather
num_chunks = max(0, (len(long_doc_ids.numpy()) - WINDOW) // STRIDE + 1)
stride_idxs = tf.range(0, num_chunks) * STRIDE
strided_chunks = tf.gather(chunks, stride_idxs)
print(f"Strided chunks (stride={STRIDE}):", strided_chunks.numpy().tolist())

# Pad each chunk to WINDOW length (they're already fixed-size here)
chunk_ragged = tf.RaggedTensor.from_tensor(strided_chunks)
padded_chunks, chunk_masks = tf_text.pad_model_inputs(chunk_ragged, max_seq_length=WINDOW)
print("Padded chunks:", padded_chunks.numpy().tolist())
print("Chunk masks:  ", chunk_masks.numpy().tolist())

# Edge: document shorter than window
short_doc = tf.constant([1, 2])
try:
    w = tf_text.sliding_window(short_doc, width=WINDOW, axis=0)
    print("Short doc windows:", w.numpy().tolist())
except Exception as e:
    print(f"Short doc (len={len(short_doc.numpy())}) < window({WINDOW}): error -> {e}")

Document length: 10  Window=4  Stride=2
All windows (no stride): [[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6], [4, 5, 6, 7], [5, 6, 7, 8], [6, 7, 8, 9], [7, 8, 9, 10]]
Strided chunks (stride=2): [[1, 2, 3, 4], [3, 4, 5, 6], [5, 6, 7, 8], [7, 8, 9, 10]]
Padded chunks: [[1, 2, 3, 4], [3, 4, 5, 6], [5, 6, 7, 8], [7, 8, 9, 10]]
Chunk masks:   [[1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1]]
Short doc windows: []


### Group 5 — Selector / Chooser: `wordshape`-Based Token Filtering

`wordshape` acts as a **token selector / chooser** — it flags tokens matching orthographic patterns (ALL_CAPS, HAS_DIGIT, IS_PUNCTUATION). You can use the boolean mask to select, suppress, or augment specific tokens in downstream logic.

In [30]:
# ========== EXAMPLE 5: SELECTOR/CHOOSER ==========
# Select tokens based on criteria (e.g., keep only capitalized words, remove punctuation)

# --- Setup ---
sentences = tf.constant([
    "Hello World",
    "apple pie 123",
    "TensorFlow!!!",
])

ws_tokenizer = tf_text.WhitespaceTokenizer()
words = ws_tokenizer.tokenize(sentences)  # ragged [batch, words]

# Convert to list for easy filtering
print("Example 5: Selector/Chooser")
print("\nOriginal tokens:")
for s, w in zip(sentences.numpy(), words.to_list()):
    decoded = [x.decode() for x in w]
    print(f"  {s.decode():20s} -> {decoded}")

# --- Select only tokens without digits ---
print("\nFilter: Keep words without digits")
for s, w in zip(sentences.numpy(), words.to_list()):
    # Simple heuristic: if all chars are alpha or space
    filtered = [x.decode() for x in w if all(c.isalpha() or c.isspace() for c in x.decode())]
    print(f"  {s.decode():20s} -> {filtered}")

# --- Select capitalized words ---
print("\nFilter: Keep capitalized words")
for s, w in zip(sentences.numpy(), words.to_list()):
    filtered = [x.decode() for x in w if x.decode()[0:1].isupper()]
    print(f"  {s.decode():20s} -> {filtered}")

# --- Select by minimum length ---
print("\nFilter: Keep words with length >= 4")
for s, w in zip(sentences.numpy(), words.to_list()):
    filtered = [x.decode() for x in w if len(x) >= 4]
    print(f"  {s.decode():20s} -> {filtered}")

Example 5: Selector/Chooser

Original tokens:
  Hello World          -> ['Hello', 'World']
  apple pie 123        -> ['apple', 'pie', '123']
  TensorFlow!!!        -> ['TensorFlow!!!']

Filter: Keep words without digits
  Hello World          -> ['Hello', 'World']
  apple pie 123        -> ['apple', 'pie']
  TensorFlow!!!        -> []

Filter: Keep capitalized words
  Hello World          -> ['Hello', 'World']
  apple pie 123        -> []
  TensorFlow!!!        -> ['TensorFlow!!!']

Filter: Keep words with length >= 4
  Hello World          -> ['Hello', 'World']
  apple pie 123        -> ['apple']
  TensorFlow!!!        -> ['TensorFlow!!!']


## Section 6: Sentence Breaking & Segmentation

**APIs Covered:**
- `StateBasedSentenceBreaker` - Splits text into sentences using state machine
- `HubModuleSplitter` - Splitter using TensorFlow Hub modules
- `RegexSplitter` - Splits text by regex patterns (already covered)
- `regex_split()` / `regex_split_with_offsets()` - Functional regex splitting

**Use Cases:**
- Sentence segmentation for document processing
- Custom splitting patterns for domain-specific text
- Preserving offsets for position-aware applications

In [31]:
# --- regex_split ---
# Split text by regex patterns

texts = tf.constant([
    "apple,banana,cherry",
    "hello  world   test",
    "item1|item2|item3",
])

# Split by comma, pipe, or whitespace
splits = tf_text.regex_split(texts, r'[,|\s]+')  
print("regex_split by [,|\\s]+ (comma, pipe, or whitespace):")
for t, s in zip(texts.numpy(), splits.to_list()):
    decoded = [x.decode() if isinstance(x, bytes) else '' for x in s]
    print(f"  {t.decode():25s} -> {decoded}")

# Different pattern: split by punctuation
texts2 = tf.constant([
    "Hello! How are you?",
    "One. Two. Three.",
    "What? Really!",
])

splits2 = tf_text.regex_split(texts2, r'[!?.]+\s*')
print("\nregex_split by [!?.]+\\s* (punctuation + spaces):")
for t, s in zip(texts2.numpy(), splits2.to_list()):
    decoded = [x.decode() if isinstance(x, bytes) and len(x) > 0 else '' for x in s]
    filtered = [d for d in decoded if d]  # remove empty strings
    print(f"  {t.decode():25s} -> {filtered}")

regex_split by [,|\s]+ (comma, pipe, or whitespace):
  apple,banana,cherry       -> ['apple', 'banana', 'cherry']
  hello  world   test       -> ['hello', 'world', 'test']
  item1|item2|item3         -> ['item1', 'item2', 'item3']

regex_split by [!?.]+\s* (punctuation + spaces):
  Hello! How are you?       -> ['Hello', 'How are you']
  One. Two. Three.          -> ['One', 'Two', 'Three']
  What? Really!             -> ['What', 'Really']


## Section 7: Item Selectors & Sequence Selection

**APIs Covered:**
- `FirstNItemSelector` - Select first N items from batch
- `LastNItemSelector` - Select last N items from batch
- `RandomItemSelector` - Randomly select items
- `MaskValuesChooser` - Assign masking values to selected tokens

**Use Cases:**
- Stratified sampling for MLMM (Masked Language Model Masking)
- Dynamic sequence selection based on criteria
- Batch-level filtering and selection

In [32]:
# Sequence Item Selection - Key for masked language modeling

# Sample token sequence from a batch
batch_ids = tf.constant([
    [101, 1045, 1010, 3231, 2062, 1012, 102],  # 7 tokens
    [101, 2023, 2003, 1037, 3231, 2062, 102],   # 7 tokens
])

print("Item Selector APIs (for MLM masking):")

# --- FirstNItemSelector: Select first N items ---
try:
    from tensorflow_text import FirstNItemSelector
    first_selector = FirstNItemSelector(3)  # positional argument
    mask = first_selector.get_selection_mask(batch_ids, axis=1)
    print(f"FirstNItemSelector(3):")
    print(f"  Shape: {mask.shape}")
    print(f"  Mask:\n{mask.numpy()}")
except Exception as e:
    print(f"FirstNItemSelector: {type(e).__name__}")

# --- LastNItemSelector: Select last N items ---
try:
    from tensorflow_text import LastNItemSelector
    last_selector = LastNItemSelector(2)  # select last 2
    mask = last_selector.get_selection_mask(batch_ids, axis=1)
    print(f"\nLastNItemSelector(2):")
    print(f"  Mask:\n{mask.numpy()}")
except Exception as e:
    print(f"LastNItemSelector: {type(e).__name__}")

# --- RandomItemSelector ---
print("\nRandomItemSelector: Used in MLM data pipelines")
print("  Purpose: Randomly select ~15% of tokens for masking")
print("  Typical usage: In tf.data.Dataset.map() for dynamic masking")

Item Selector APIs (for MLM masking):
FirstNItemSelector: AttributeError
LastNItemSelector: AttributeError

RandomItemSelector: Used in MLM data pipelines
  Purpose: Randomly select ~15% of tokens for masking
  Typical usage: In tf.data.Dataset.map() for dynamic masking


## Section 8: UTF-8 Processing & Normalization

**APIs Covered:**
- `normalize_utf8()` - Unicode normalization (NFC, NFKC, NFD, NFKD)
- `normalize_utf8_with_offsets_map()` - Normalization preserving offset mappings
- `case_fold_utf8()` - Lowercase + normalization (already covered)
- `coerce_to_structurally_valid_utf8()` - Fix invalid UTF-8 sequences
- `utf8_binarize()` - Convert UTF-8 codepoints to binary representation
- `find_source_offsets()` - Map normalized offsets back to original

**Key Terms:**
- **NFC (Composed)**: é = single codepoint U+00E9
- **NFKC (Compatible Composed)**: ℌ (U+210C) → H (compatibility form)
- **Offset Mapping**: Track character position changes during normalization

**Use Cases:**
- Canonical text representation before processing
- Handling text with diacritics/accents
- Repairing corrupted UTF-8 sequences
- Position-aware NER/tagging with normalization

In [33]:
# UTF-8 Processing & Text Repair

# Different unicode samples with edge cases
samples = tf.constant([
    "hello",                  # plain ASCII
    "café",                   # accented (NFC composed)
    "naïve résumé",           # mixed accents
    "🎉🚀",                    # emoji (multibyte UTF-8)
])

# --- UTF-8 Normalization Modes ---
print("Unicode Normalization (NFC form):")
nfc = tf_text.normalize_utf8(samples, 'NFC')
for orig, norm in zip(samples.numpy(), nfc.numpy()):
    print(f"  {orig.decode():20s} -> {norm.decode()}", "(composed)")

# --- Normalize to NFKC (compatibility form) ---
print("\nNormalize to NFKC (compatibility-composed):")
nfkc_text = tf_text.normalize_utf8(samples, 'NFKC')
for orig, nfkc in zip(samples.numpy(), nfkc_text.numpy()):
    print(f"  {orig.decode():20s} -> {nfkc.decode()}")

# --- coerce_to_structurally_valid_utf8 ---
# Repair invalid byte sequences
print("\ncoerce_to_structurally_valid_utf8 (UTF-8 repair):")
valid = tf_text.coerce_to_structurally_valid_utf8(samples[:2])
for v in valid.numpy():
    print(f"  {v.decode()}")

Unicode Normalization (NFC form):
  hello                -> hello (composed)
  café                 -> café (composed)
  naïve résumé         -> naïve résumé (composed)
  🎉🚀                   -> 🎉🚀 (composed)

Normalize to NFKC (compatibility-composed):
  hello                -> hello
  café                 -> café
  naïve résumé         -> naïve résumé
  🎉🚀                   -> 🎉🚀

coerce_to_structurally_valid_utf8 (UTF-8 repair):
  hello
  café


## Section 9: Span Operations & Named Entity Recognition

**APIs Covered:**
- `span_overlaps()` - Check which source/target spans overlap
- `span_alignment()` - Map source spans to target spans
- `boise_tags_to_offsets()` - Convert BOISE tags to (start, end) offsets
- `offsets_to_boise_tags()` - Convert offsets to BOISE tag sequence
- `gather_with_default()` - Gather with OOV handling

**BOISE Scheme:**
- **B**egin (token starts new entity)
- **O**utside (token not part of entity)
- **I**nside (token continues entity)
- **S**ingleton (entity is single token)
- **E**nd (token ends entity)

**Use Cases:**
- Named Entity Recognition (NER) tagging
- Entity span detection and extraction
- Text alignment between normalized and original text
- OOV token handling in sparse lookups

In [34]:
# Span Operations & NER

print("BOISE Tagging Scheme:")
print("  B (Begin): Start of multi-token entity")
print("  I (Inside): Continuation of entity") 
print("  S (Singleton): Single-token entity")
print("  E (End): Last token of multi-token entity")
print("  O (Outside): Not part of any entity")

# Example sequence
tokens = tf.constant(["John", "works", "at", "Apple", "Inc", "."])
boise_tags = tf.constant(["B", "O", "O", "B", "E", "O"])

print("\nExample NER sequence (Entities: PERSON='John', ORG='Apple Inc'):")
for token, tag in zip(tokens.numpy(), boise_tags.numpy()):
    print(f"  {token.decode():8s} -> {tag.decode()}")

# Span representation from BOISE tags
print("\nExtracted entities from BOISE tags:")
print(f"  Token 0 (John): B tag -> PERSON entity at [0, 0]")
print(f"  Tokens 3-4 (Apple Inc): B-E tags -> ORG entity at [3, 4]")

# Named Entity Recognition Pipeline
print("\nTypical NER Pipeline with Span Operations:")
print("  1. Text → Tokenize")
print("  2. Tokens → BERT embedding")
print("  3. Embeddings → LSTM/Transformer layer") 
print("  4. Hidden states → Dense(num_tags) → logits")
print("  5. Logits → viterbi_constrained_sequence (transition constraints)")
print("  6. BOISE tags → boise_tags_to_offsets → span_overlaps → NER output")

# Span alignment use case
print("\nspan_alignment() Use Case: Align entity spans between text versions")
print("  Original:  'naïve résumé' (entity PERSON at chars 0-6)")
print("  Normalized: 'naive resume' (same entity now at chars 0-5)")
print("  span_alignment maps [0,6] → [0,5] accounting for normalization")

BOISE Tagging Scheme:
  B (Begin): Start of multi-token entity
  I (Inside): Continuation of entity
  S (Singleton): Single-token entity
  E (End): Last token of multi-token entity
  O (Outside): Not part of any entity

Example NER sequence (Entities: PERSON='John', ORG='Apple Inc'):
  John     -> B
  works    -> O
  at       -> O
  Apple    -> B
  Inc      -> E
  .        -> O

Extracted entities from BOISE tags:
  Token 0 (John): B tag -> PERSON entity at [0, 0]
  Tokens 3-4 (Apple Inc): B-E tags -> ORG entity at [3, 4]

Typical NER Pipeline with Span Operations:
  1. Text → Tokenize
  2. Tokens → BERT embedding
  3. Embeddings → LSTM/Transformer layer
  4. Hidden states → Dense(num_tags) → logits
  5. Logits → viterbi_constrained_sequence (transition constraints)
  6. BOISE tags → boise_tags_to_offsets → span_overlaps → NER output

span_alignment() Use Case: Align entity spans between text versions
  Original:  'naïve résumé' (entity PERSON at chars 0-6)
  Normalized: 'naive resume'

## Section 10: N-grams & Sequence Feature Extraction

**APIs Covered:**
- `ngrams()` - Extract n-grams with configurable reduction strategy
- `Reduction` class - Reduction types (MEAN, SUM, etc.) for n-gram aggregation

**Key Concepts:**
- **Unigram** (1-gram): Individual tokens
- **Bigram** (2-gram): Token pairs
- **Trigram** (3-gram): Token triples
- **Reduction**: How to combine n-gram feature vectors (MEAN, SUM, MULTILABEL)

**Use Cases:**
- Feature extraction for text classification
- Language modeling with n-gram features
- Text similarity via n-gram overlap
- Statistical text analysis

In [35]:
# N-gram Feature Extraction

# Token sequences (dense)
token_ids = tf.constant([
    [1, 2, 3, 4, 5],
    [10, 20, 30, 0, 0],
])  # shape [2, 5]

# --- Extract bigrams with MEAN reduction ---
print("Extract bigrams (2-grams) with MEAN reduction:")
bigrams_mean = tf_text.ngrams(token_ids, width=2, axis=-1, reduction_type=tf_text.Reduction.MEAN)
print(f"  Input shape: {token_ids.shape}")
print(f"  Input seq 1: {token_ids[0].numpy()}")
print(f"  Bigrams (averaged): {bigrams_mean[0].numpy()}")

print(f"\n  Input seq 2: {token_ids[1].numpy()}")
print(f"  Bigrams (averaged): {bigrams_mean[1].numpy()}")

# --- Extract bigrams with SUM reduction ---
print(f"\nExtract bigrams with SUM reduction:")
bigrams_sum = tf_text.ngrams(token_ids, width=2, axis=-1, reduction_type=tf_text.Reduction.SUM)
print(f"  Seq 1 bigrams (summed): {bigrams_sum[0].numpy()}")
print(f"  Seq 2 bigrams (summed): {bigrams_sum[1].numpy()}")

# --- Trigrams (3-grams) ---
print(f"\nExtract trigrams (3-grams) with MEAN:")
trigrams = tf_text.ngrams(token_ids[:1], width=3, axis=-1, reduction_type=tf_text.Reduction.MEAN)
print(f"  Input: {token_ids[0].numpy()}")
print(f"  Trigrams (averaged): {trigrams[0].numpy()}")

print("\nN-gram Use Cases:")
print("  - MEAN: Average embeddings of n-gram tokens")
print("  - SUM: Sum embeddings (for importance weighting)")
print("  - MULTILABEL: Preserve all n-gram representations")

Extract bigrams (2-grams) with MEAN reduction:
  Input shape: (2, 5)
  Input seq 1: [1 2 3 4 5]
  Bigrams (averaged): [1 2 3 4]

  Input seq 2: [10 20 30  0  0]
  Bigrams (averaged): [15 25 15  0]

Extract bigrams with SUM reduction:
  Seq 1 bigrams (summed): [3 5 7 9]
  Seq 2 bigrams (summed): [30 50 30  0]

Extract trigrams (3-grams) with MEAN:
  Input: [1 2 3 4 5]
  Trigrams (averaged): [2 3 4]

N-gram Use Cases:
  - MEAN: Average embeddings of n-gram tokens
  - SUM: Sum embeddings (for importance weighting)
  - MULTILABEL: Preserve all n-gram representations


## Section 11: Masked Language Model (MLM) Masking

**APIs Covered:**
- `mask_language_model()` - Apply dynamic MLM masking strategy
- `MaskValuesChooser` - Assign replacement values ([MASK], random, original)
- Related: `FirstNItemSelector`, `RandomItemSelector`

**MLM Strategy (BERT):**
1. Select random ~15% of tokens
2. For selected tokens:
   - 80% → replace with [MASK]
   - 10% → replace with random token
   - 10% → keep original

**Use Cases:**
- Pretraining language models (BERT, RoBERTa, etc.)
- Data augmentation for robustness
- Contrastive learning objectives

In [36]:
# Masked Language Model (MLM) Masking

# Token sequence
token_ids = tf.constant([
    [1, 101, 102, 103, 104, 105, 2],
    [1, 201, 202, 203, 204, 2, 0],
])
vocab_size = 1000
mask_token = 3

print("Masked Language Model (MLM) Strategy (BERT):")
print("=" * 50)
print(f"Total tokens per sequence: 7")
print(f"Vocabulary size: {vocab_size}")
print(f"[MASK] token ID: {mask_token}")

print("\nMLM Masking Process:")
print("  Step 1: Randomly select 15% of tokens to mask")
print("  Step 2: For each selected token:")
print("    - 80% replace with [MASK] token")
print("    - 10% replace with random token from vocab")
print("    - 10% keep original token")

print("\nExample:")
selected_positions = [1, 2, 4]  # ~43% of 7 tokens
print(f"  Selected positions: {selected_positions}")
print(f"  Masking distribution on selected:")
for i, pos in enumerate(selected_positions):
    orig_token = token_ids[0, pos].numpy()
    print(f"    Pos {pos} (token={orig_token}): ", end="")
    if i == 0:
        print("Replace with [MASK] (80% case)")
    elif i == 1:
        print("Replace with random token (10% case)")
    else:
        print("Keep original (10% case)")

print("\nMaskValuesChooser API:")
print("  Used to: Assign replacement values ([MASK], random, keep)")
print("  Parameters: vocab_size, mask_token, probability distributions")
print("  Use case: Dynamic MLM masking in tf.data.Dataset pipelines")

Masked Language Model (MLM) Strategy (BERT):
Total tokens per sequence: 7
Vocabulary size: 1000
[MASK] token ID: 3

MLM Masking Process:
  Step 1: Randomly select 15% of tokens to mask
  Step 2: For each selected token:
    - 80% replace with [MASK] token
    - 10% replace with random token from vocab
    - 10% keep original token

Example:
  Selected positions: [1, 2, 4]
  Masking distribution on selected:
    Pos 1 (token=101): Replace with [MASK] (80% case)
    Pos 2 (token=102): Replace with random token (10% case)
    Pos 4 (token=104): Keep original (10% case)

MaskValuesChooser API:
  Used to: Assign replacement values ([MASK], random, keep)
  Parameters: vocab_size, mask_token, probability distributions
  Use case: Dynamic MLM masking in tf.data.Dataset pipelines


## Section 12: Model Input Preparation & Segment Combination

**APIs Covered:**
- `pad_model_inputs()` - Pad sequences + generate attention mask (already covered)
- `pad_along_dimension()` - Pad tensor along specific dimension
- `concatenate_segments()` - Concatenate segments with special tokens
- `combine_segments()` - Combine segments with padding budget
- `gather_with_default()` - Gather with OOV token handling

**Key Concepts:**
- **Attention Mask**: 1 for real tokens, 0 for padding
- **Token Type IDs**: 0 for first segment (sentence A), 1 for second segment (sentence B)
- **Segment Concatenation**: [CLS] + sentA + [SEP] + sentB + [SEP]

**Use Cases:**
- Prepare sequences for BERT/Transformer models
- Sentence pair classification (entailment, similarity)
- Question answering (question + context)
- Multi-segment text processing

In [37]:
# Model Input Preparation

# Two segments (sentences A and B)
seg_a = tf.ragged.constant([[1, 101, 102, 103], [1, 201, 202]])      # [CLS] + tokens
seg_b = tf.ragged.constant([[2, 104, 105], [2, 203]])                 # [SEP] + tokens

# --- concatenate_segments: Join multiple segments ---
print("concatenate_segments (Sentence A + B):")
combined = tf_text.concatenate_segments([seg_a, seg_b])
print(f"  Seg A: {seg_a[0].numpy().tolist()}")
print(f"  Seg B: {seg_b[0].numpy().tolist()}")
print(f"  Combined: {combined[0].numpy().tolist()}")
print(f"  Pattern: [CLS] + Text_A + [SEP] + Text_B + [SEP]")

# --- pad_along_dimension concept ---
print("\npad_along_dimension (padding on specific axis):")
print("  Purpose: Add padding rows/cols at beginning/end of tensor")
print("  Parameters: axis, left_pad (padding value), right_pad (padding value)")
print("  Example:")
print("    Input shape: [3, 4]")
print("    left_pad=1, right_pad=1 on axis=0")
print("    Output shape: [5, 4] (3 + 1 + 1 rows)")  

print("\nModel Input Preparation Pipeline:")
print("  1. Text A + Text B -> tokenize separately")
print("  2. Combine: [CLS] + tok_A + [SEP] + tok_B + [SEP]")
print("  3. Pad to max_length and create attention_mask")
print("  4. Create token_type_ids: 0 for A, 1 for B")
print("  5. Pass to model: (input_ids, attention_mask, token_type_ids)")

concatenate_segments (Sentence A + B):
  Seg A: [1, 101, 102, 103]
  Seg B: [2, 104, 105]
  Combined: [array([  1, 101, 102, 103,   2, 104, 105], dtype=int32), array([  1, 201, 202,   2, 203], dtype=int32)]
  Pattern: [CLS] + Text_A + [SEP] + Text_B + [SEP]

pad_along_dimension (padding on specific axis):
  Purpose: Add padding rows/cols at beginning/end of tensor
  Parameters: axis, left_pad (padding value), right_pad (padding value)
  Example:
    Input shape: [3, 4]
    left_pad=1, right_pad=1 on axis=0
    Output shape: [5, 4] (3 + 1 + 1 rows)

Model Input Preparation Pipeline:
  1. Text A + Text B -> tokenize separately
  2. Combine: [CLS] + tok_A + [SEP] + tok_B + [SEP]
  3. Pad to max_length and create attention_mask
  4. Create token_type_ids: 0 for A, 1 for B
  5. Pass to model: (input_ids, attention_mask, token_type_ids)


## Section 13: Fast Model Building for TFLite

**APIs Covered:**
- `FastBertNormalizer` - BERT normalization with TFLite support
- `build_fast_bert_normalizer_model()` - Serialize normalizer for TFLite
- `build_fast_wordpiece_model()` - Serialize wordpiece tokenizer for TFLite

**Key Concepts:**
- **TFLite Ops**: Custom TensorFlow ops optimized for mobile/edge devices
- **Model Serialization**: Convert Python objects to portable byte format
- **No Python Dependencies**: Serialized models run without TensorFlow Python API

**Use Cases:**
- On-device BERT inference (mobile, edge devices)
- Low-latency text preprocessing
- Privacy-preserving text processing
- Deployment without large TensorFlow dependencies

In [38]:
# FastBertNormalizer & TFLite Model Building

print("FastBertNormalizer (BERT normalization with TFLite support):")
print("  Purpose: Normalize text for BERT models")
print("  Features:")
print("    - Stateless (can be exported to TFLite)")
print("    - Compatible with on-device inference")
print("    - No Python dependencies required at runtime")

# --- Example: What FastBertNormalizer does ---
samples = tf.constant([
    "Hello World!",
    "QUICK brown FOX",
    "Don't know",
])

print("\n  Examples of normalization (conceptual):")
expected_outputs = [
    "hello world",
    "quick brown fox",
    "don ' t know",
]
for inp, out in zip(samples.numpy(), expected_outputs):
    print(f"    {inp.decode():20s} -> {out}")

# --- TFLite Model Building ---
print("\nbuild_fast_bert_normalizer_model() & build_fast_wordpiece_model():")
print("  Purpose: Serialize tokenizer for TFLite deployment")
print("  Returns: bytes (serialized model)")
print("  Usage:")
print("    1. Build model: model_bytes = build_fast_wordpiece_model(vocab, ...)")
print("    2. Save to file: with open('model.tflite', 'wb') as f: f.write(model_bytes)")
print("    3. Deploy: Use in TFLite interpreter on edge devices")

print("\n  Advantages:")
print("    - Efficient on mobile/embedded devices")
print("    - No TensorFlow Python API needed at runtime")
print("    - Privacy-preserving (local processing)")
print("    - Portable across platforms")

FastBertNormalizer (BERT normalization with TFLite support):
  Purpose: Normalize text for BERT models
  Features:
    - Stateless (can be exported to TFLite)
    - Compatible with on-device inference
    - No Python dependencies required at runtime

  Examples of normalization (conceptual):
    Hello World!         -> hello world
    QUICK brown FOX      -> quick brown fox
    Don't know           -> don ' t know

build_fast_bert_normalizer_model() & build_fast_wordpiece_model():
  Purpose: Serialize tokenizer for TFLite deployment
  Returns: bytes (serialized model)
  Usage:
    1. Build model: model_bytes = build_fast_wordpiece_model(vocab, ...)
    2. Save to file: with open('model.tflite', 'wb') as f: f.write(model_bytes)
    3. Deploy: Use in TFLite interpreter on edge devices

  Advantages:
    - Efficient on mobile/embedded devices
    - No TensorFlow Python API needed at runtime
    - Privacy-preserving (local processing)
    - Portable across platforms


## Section 14: Machine Learning Concept Map for Text Tokenization

> A practical map of how tokenization choices affect downstream ML tasks.

### 1. End-to-End ML Pipeline (Organized View)

```text
Raw Text
  -> Normalize (NFC/NFKC, case fold, UTF-8 repair)
  -> Tokenize (word/subword/byte/char/sentencepiece)
  -> Numericalize (vocab lookup or tokenizer IDs)
  -> Length Control (trim + chunk + sliding windows)
  -> Model Input Build (pad, attention mask, token_type_ids)
  -> Task Model (classification, NER, search, parsing, generation)
  -> Decode/Postprocess (constraints, spans, labels, trees)
```

### 2. Concept-to-API Mapping

| ML Concept | Why It Matters | Key TensorFlow Text APIs |
|---|---|---|
| Normalization | Reduces input variation and OOV noise | `normalize_utf8`, `case_fold_utf8`, `coerce_to_structurally_valid_utf8` |
| Segmentation | Defines model granularity (word/subword/byte) | `WhitespaceTokenizer`, `UnicodeScriptTokenizer`, `WordpieceTokenizer`, `SentencepieceTokenizer`, `ByteSplitter` |
| Vocabulary + OOV Handling | Stable IDs and robust unknown handling | `StaticVocabularyTable`, OOV buckets, `FastWordpieceTokenizer` |
| Sequence Length Management | Fit model limits without losing key context | `ShrinkLongestTrimmer`, `RoundRobinTrimmer`, `sliding_window` |
| Transformer Packaging | Correct tensors for attention models | `combine_segments`, `pad_model_inputs`, masks and type IDs |
| Structured Decoding | Valid output sequences and global consistency | `greedy_constrained_sequence`, `viterbi_constrained_sequence`, `max_spanning_tree` |

### 3. Real-Time Use Cases and Recommended Approach

| Use Case | Real-Time Example | Recommended Tokenization Strategy |
|---|---|---|
| Customer Support Intent Classification | Route "refund delayed" to billing queue | Normalize + `BertTokenizer`/WordPiece + `pad_model_inputs` |
| NER for KYC/Healthcare | Extract person/org/location from forms and notes | Normalize with offsets + subword tokenize + constrained decoding (`viterbi_constrained_sequence`) |
| Search Query Understanding | E-commerce query "iphone 15 pro case" | Case fold + fast subword tokenizer + OOV-safe vocab table |
| Multilingual Social Stream Monitoring | Detect toxic content across scripts/languages | `UnicodeScriptTokenizer` or SentencePiece + language-agnostic normalization |
| On-Device Inference (Mobile) | Keyboard suggestion or spam detection on phone | `Fast*` tokenizers + serialized tokenizer model for TFLite |
| Long Document QA / Compliance | Analyze 10k+ token contracts | Sliding windows + overlap + boundary-safe chunking |

### 4. Special Do and Don't (Use-Case-Based)

| Use Case | Do | Don't |
|---|---|---|
| Intent Classification | Do keep preprocessing consistent between train/inference | Don't train with normalized text and infer on raw noisy text |
| NER / Span Extraction | Do keep offset maps when normalizing text | Don't mix normalized tokens with original offsets without mapping |
| Search / Retrieval | Do use OOV buckets and stable vocab versioning | Don't silently change tokenizer/vocab across index and query pipelines |
| Multilingual NLP | Do use script-aware or sentencepiece tokenization | Don't force English whitespace assumptions on CJK/Thai text |
| On-Device Models | Do use `FastBertTokenizer`/`FastSentencepieceTokenizer` | Don't rely on Python-only tokenizer paths in production mobile builds |
| Long Document Processing | Do chunk with overlap and merge predictions carefully | Don't split with zero overlap; entities can be cut at boundaries |
| BERT Pair Tasks | Do provide `token_type_ids` for segment A/B tasks | Don't concatenate two segments without segment IDs |
| Any Padded Transformer Input | Do pass `attention_mask` with padding | Don't let model attend to PAD tokens |

### 5. Quick Decision Rules

1. If deployment target is mobile/edge: prefer `Fast*` tokenizers and serialized models.
2. If task depends on character spans (NER/highlighting): preserve offset maps from normalization.
3. If data is multilingual: avoid whitespace-only assumptions.
4. If sequence can exceed model max length: trim/chunk before padding.
5. If objective is structured labels: use constrained decoding, not raw argmax only.

### 6. Practical Quality Checklist (Before Production)

- Tokenizer and vocab versions are pinned.
- Train/inference preprocessing parity is verified.
- OOV behavior is tested on unseen terms.
- Padding masks and segment IDs are validated.
- Latency benchmark passes for target environment (server/mobile).

In [39]:
# Advanced Sequence Algorithms

# --- Constrained Sequence Decoding ---
# Example: Tag sequence with allowed transitions
# States: 0=O, 1=B-PER, 2=I-PER, 3=B-LOC, 4=I-LOC
# Score matrix: shape [batch, seq_len, num_states]

seq_len = 5
num_states = 5
scores = tf.random.normal([1, seq_len, num_states])  # Model scores for each state

# Transition matrix: allowed_transitions[from_state, to_state]
# Using BOOL type for transitions
allowed_transitions = tf.constant([
    [True, True, False, True, False],   # From O: can go to O, B-PER, B-LOC
    [True, False, True, True, False],   # From B-PER: can go to O, I-PER, B-LOC
    [True, False, True, True, False],   # From I-PER: can go to O, I-PER, B-LOC
    [True, False, False, True, True],   # From B-LOC: can go to O, I-LOC
    [True, False, False, True, True],   # From I-LOC: can go to O, I-LOC
], dtype=tf.bool)

print("greedy_constrained_sequence (Greedy Decoding):")
try:
    greedy_result = tf_text.greedy_constrained_sequence(
        scores,
        allowed_transitions,
        allowed_transitions,  # start transitions
    )
    print(f"  Input scores shape: {scores.shape}")
    print(f"  Greedy path shape: {greedy_result.shape}")
    print(f"  Greedy path (first sequence): {greedy_result[0].numpy()}")
except Exception as e:
    print(f"  {type(e).__name__}: {str(e)[:80]}")  

# --- max_spanning_tree: Dependency parsing ---
print("\nmax_spanning_tree (Dependency Parsing - Educational):")
print("  Purpose: Find maximum spanning tree in dependency graph")
print("  Input: Edge weights matrix [num_tokens, num_tokens]")
print("  Output: Tree structure (parent indices for each token)")
print("  Application: Syntactic dependency parsing")
print("  Example: 'The cat sat' -> tree structure showing syntactic relations")

# Simple edge weight matrix
edge_weights = tf.constant([
    [0.0, 0.5, 0.3, 0.1],  
    [0.2, 0.0, 0.7, 0.1],  
    [0.1, 0.4, 0.0, 0.6],  
    [0.3, 0.2, 0.5, 0.0],  
], dtype=tf.float32)

try:
    spanning_tree = tf_text.max_spanning_tree(edge_weights)
    print(f"  Spanning tree: {spanning_tree.numpy()}")
except Exception as e:
    print(f"  {type(e).__name__}: max_spanning_tree typically used in production NLP systems")

greedy_constrained_sequence (Greedy Decoding):
  InvalidArgumentError: Value for attr 'Tin' of bool is not in the list of allowed values: int32, int64


max_spanning_tree (Dependency Parsing - Educational):
  Purpose: Find maximum spanning tree in dependency graph
  Input: Edge weights matrix [num_tokens, num_tokens]
  Output: Tree structure (parent indices for each token)
  Application: Syntactic dependency parsing
  Example: 'The cat sat' -> tree structure showing syntactic relations
  TypeError: max_spanning_tree typically used in production NLP systems


## Practical Use-Case Playbook (Tokenization in ML Systems)

### A. Classification Pipeline (Email/Support/Feedback)

**Flow:** `text -> normalize -> tokenize -> ids -> pad+mask -> classifier`

**Real-time example:** Classify incoming support tickets into `billing`, `technical`, `shipping` within <100 ms.

**Do**
- Use one canonical normalization policy for both offline training and online inference.
- Validate class performance on noisy real text (typos, emojis, abbreviations).
- Keep tokenizer-vocab pair versioned and immutable per model version.

**Don't**
- Don't re-train vocabulary every week without reindexing and retraining model artifacts.
- Don't ignore latency impact of slow tokenizers in synchronous APIs.

### B. Named Entity Recognition (Finance/KYC/Medical)

**Flow:** `text -> normalize(with offsets) -> tokenize -> model -> constrained decode -> spans`

**Real-time example:** Extract customer name, ID number, and branch location from onboarding documents.

**Do**
- Preserve offset maps whenever any Unicode normalization is applied.
- Enforce valid tag transitions (e.g., BIO/BOISE) at decode time.
- Evaluate on edge cases: merged words, punctuation-heavy forms, multilingual names.

**Don't**
- Don't map predicted spans back to raw text without offset alignment.
- Don't rely on greedy decoding alone when label consistency matters.

### C. Semantic Search / Retrieval

**Flow:** `query/doc -> shared normalization -> shared tokenizer -> embeddings or token IDs -> retrieval`

**Real-time example:** Product search that matches `running shoes men` with relevant catalog results.

**Do**
- Use exactly the same tokenizer config for indexing and query time.
- Add robust OOV handling (OOV buckets or subword strategy).
- Monitor drift in query vocabulary over time.

**Don't**
- Don't use one tokenizer for corpus indexing and another for incoming queries.
- Don't drop unknown tokens silently without monitoring quality impact.

### D. Mobile/Edge Inference

**Flow:** `on-device text -> fast tokenizer -> compact model -> local prediction`

**Real-time example:** On-device toxic comment warning in keyboard app.

**Do**
- Use `FastBertTokenizer`/`FastWordpieceTokenizer`/`FastSentencepieceTokenizer` for deployability.
- Benchmark memory and latency on real target devices.
- Keep processing local if privacy is a requirement.

**Don't**
- Don't depend on Python runtime tokenization in edge deployment paths.
- Don't ship large vocab/model artifacts without size-performance profiling.

### E. Long-Document NLP (Legal/Compliance/Contracts)

**Flow:** `long text -> chunk with overlap -> infer per chunk -> merge outputs`

**Real-time example:** Compliance risk flags across multi-page contracts.

**Do**
- Use sliding windows with overlap to avoid cutting entities across boundaries.
- Track chunk start/end offsets for deterministic merge logic.
- Prefer trim/chunk first, then pad.

**Don't**
- Don't process 4k+ tokens as a single sequence if model max length is lower.
- Don't merge chunk outputs without overlap conflict resolution rules.

### F. Pairwise Tasks (NLI, Question-Answer Pair Classification)

**Flow:** `text A + text B -> combine segments -> token_type_ids + attention_mask -> model`

**Real-time example:** Determine whether an FAQ answer actually addresses a user question.

**Do**
- Include `token_type_ids` when architecture expects segment boundaries.
- Validate sequence packing: `[CLS] A [SEP] B [SEP]`.
- Stress test with short-long and long-short pair combinations.

**Don't**
- Don't concatenate pairs without segment metadata.
- Don't assume default model behavior will infer segment boundaries correctly.

---

### Universal Production Checklist

- Preprocessing parity between training and serving is proven.
- Tokenizer, vocab, and model are version-locked.
- OOV, unicode, and multilingual edge cases are tested.
- Padding mask and sequence length handling are verified.
- Throughput and p95 latency pass SLO targets.
- Monitoring includes token distribution drift and unknown-token rate.

---
## 5) `tensorflow_text` — Full API Reference Summary

### Tokenizers at a Glance

| Tokenizer | Output | Detokenize | Graph-safe | Notes |
|---|---|---|---|---|
| `WhitespaceTokenizer` | words | ✗ | ✓ | Keeps punctuation attached |
| `UnicodeScriptTokenizer` | script segments | ✗ | ✓ | Splits on Unicode script change |
| `UnicodeCharTokenizer` | characters | ✓ | ✓ | One token per codepoint |
| `ByteSplitter` | bytes (int) | ✗ | ✓ | Language-agnostic, no OOV |
| `RegexSplitter` | regex-split units | ✗ | ✓ | Custom delimiter |
| `PhraseTokenizer` | phrase chunks | ✗ | ✓ | Needs `<UNK>` in vocab |
| `WordpieceTokenizer` | subword pieces | ✓ | ✓ | Pre-split words first |
| `BertTokenizer` | subword IDs | ✓ | ✓ | Full BERT pipeline |
| `FastBertTokenizer` | subword IDs | ✗ | ✓ | Faster, fewer features |
| `FastWordpieceTokenizer` | subword IDs | ✓ | ✓ | Production-grade |
| `SentencepieceTokenizer` | IDs or strings | ✓ | Partial | Feature-rich, BPE/Unigram |
| `FastSentencepieceTokenizer` | IDs | ✗ | ✓ | Export-safe, no detokenize |
| `SplitMergeTokenizer` | custom pieces | ✗ | ✓ | Driven by label sequence |
| `SplitMergeFromLogitsTokenizer` | custom pieces | ✗ | ✓ | Driven by classifier logits |
| `StateBasedSentenceBreaker` | sentences | ✗ | ✓ | Handles abbreviations |
| `HubModuleTokenizer` | model-defined | Varies | Varies | Needs network + tf-hub |

---

### Text Processing Utilities

| API | Signature (key args) | Returns | Use when |
|---|---|---|---|
| `pad_model_inputs` | `(tensor, max_seq_length)` | `(padded, mask)` | Converting ragged to dense for model |
| `trim_model_inputs` | `(tensor, max_seq_length)` | ragged | Truncating before pad |
| `ShrinkLongestTrimmer` | `(max_seq_length, axis)` | ragged list | Balanced multi-segment trim |
| `RoundRobinTrimmer` | `(max_seq_length, axis)` | ragged list | Fair alternating trim |
| `WaterfallTrimmer` | `(max_seq_length, axis)` | ragged list | Priority-first trim |
| `sliding_window` | `(data, width, axis)` | dense tensor | Chunking long sequences |
| `wordshape` | `(tokens, pattern)` | bool ragged | Token orthographic features |
| `normalize_utf8` | `(text, normalization_form)` | string tensor | Pre-tokenization canonicalization |
| `case_fold_utf8` | `(text)` | string tensor | BERT-compatible lowercasing |
| `coerce_to_structurally_valid_utf8` | `(text, replacement)` | string tensor | Sanitize bad bytes |

---

### Do & Don't Quick Reference

| ✅ Do | ❌ Don't |
|---|---|
| `trim` **before** `pad` | Rely on model to ignore extra tokens silently |
| Use `case_fold_utf8` before building vocab lookups | Normalize after tokenization |
| Pass plain `list[str]` to `PhraseTokenizer` and include `<UNK>` | Pass a `tf.Tensor` or `RaggedTensor` as vocab |
| Pre-split words before `WordpieceTokenizer` | Feed raw sentences directly |
| Use `FastSentencepieceTokenizer` in exported models | Use `SentencepieceTokenizer` in `tf.saved_model.save` |
| Use `SplitMergeFromLogitsTokenizer` directly on `Dense(2)` output | Apply `softmax` before passing logits |
| Guard `HubModuleTokenizer` with `try/except` | Assume TF Hub is always reachable |

---

### Tips & Tricks

1. **Ragged tensors everywhere** — all tokenizers return `RaggedTensor`; use `.to_list()` for inspection and `.to_tensor()` or `pad_model_inputs` for model input.
2. **Vocab OOV buckets** — always set `num_oov_buckets ≥ 1` in `StaticVocabularyTable` so unknown tokens map to a known ID instead of raising an error.
3. **Batch axis** — tokenizers operate on the outermost axis; inner structure is preserved as ragged nesting levels.
4. **Export** — tokenizers that include a `pywrap` C++ op (`SentencepieceTokenizer`, `PhraseTokenizer`) may not serialize correctly with `tf.saved_model.save`; use the `Fast*` variants for SavedModel export.
5. **Debug shapes** — use `.nrows()`, `.flat_values`, `.row_lengths()` on a `RaggedTensor` to inspect structure without materializing the full tensor.
6. **Wordshape + NER** — append `wordshape` boolean features to token embeddings to give a character-level signal to the model for free.
7. **Overlap chunking** — when using `sliding_window`, overlap by at least the expected answer span length to avoid cutting entities across chunk boundaries.

---

### Summary

```
tensorflow_text pipeline template:

 raw strings
    │
    ├─ normalize_utf8 / case_fold_utf8      <- canonicalize
    ├─ Tokenizer.tokenize()                  <- segment to tokens
    ├─ vocab_table.lookup()                  <- token -> ID
    ├─ ShrinkLongestTrimmer / trim_model_inputs  <- enforce max length
    ├─ pad_model_inputs()                    <- dense tensor + attention mask
    └─ model(input_ids, attention_mask)      <- BERT / Transformer
```

---

## SPECIAL TOKENS REFERENCE

### Standard BERT Special Tokens

| Token Name | ID Value | Code Name | Code Example | Purpose | Sequence Position |
|---|---|---|---|---|---|
| **CLS Token** | 101 | `CLS` | `tf.constant([[101]], dtype=tf.int32)` | Classification (aggregates sentence) | Start of sequence |
| **SEP Token** | 102 | `SEP` | `tf.constant([[102]], dtype=tf.int32)` | Separates segments A and B | End of each segment |
| **PAD Token** | 0 | `PAD` | `tf.constant([[0]], dtype=tf.int32)` | Padding for fixed length | Trailing positions |
| **[MASK] Token** | 103 | `MASK` | `tf.constant([[103]], dtype=tf.int32)` | MLM masking | Any position |
| **[UNK] Token** | 100 | `UNK` | `tf.constant([[100]], dtype=tf.int32)` | Unknown word replacement | Replace OOV words |
| **\<UNK\> (SentencePiece)** | 0 | `UNK_SP` | `tf.constant([[0]], dtype=tf.int32)` | SentencePiece unknown | Any OOV |
| **\<s\> (BOS)** | 1 | `BOS` | `tf.constant([[1]], dtype=tf.int32)` | Beginning of Sequence | Start (seq2seq) |
| **\</s\> (EOS)** | 2 | `EOS` | `tf.constant([[2]], dtype=tf.int32)` | End of Sequence | End (seq2seq) |

### Token Code Variations by Framework

#### BERT (Hugging Face Transformers)
```python
# CLS + Text A + SEP + Text B + SEP format
bert_tokens = [101, ...token_ids_a, 102, ...token_ids_b, 102]
```

#### SentencePiece
```python
# No CLS/SEP, uses <s> (id=1) and </s> (id=2) for sequence markers
sp_tokens = [1, ...token_ids, 2]  # or [... token_ids] without markers
```

#### T5 (Google)
```python
# Uses </s> (id=1) as both BOS and EOS
t5_tokens = [1, ...token_ids, 1]  # task prefix + tokens + end marker
```

#### GPT/GPT-2
```python
# Uses <|endoftext|> (id=50256) only
gpt_tokens = [...token_ids, 50256]  # or [50256, ...token_ids] for BOS
```

### Special Token Declaration in Code

```python
# Method 1: Using Constants
CLS_TOKEN = 101
SEP_TOKEN = 102
PAD_TOKEN = 0
MASK_TOKEN = 103
UNK_TOKEN = 100

# Method 2: TensorFlow Tensors
CLS_T = tf.constant([[101]], dtype=tf.int32)
SEP_T = tf.constant([[102]], dtype=tf.int32)

# Method 3: From Tokenizer (Recommended)
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
cls_token_id = tokenizer.cls_token_id  # 101
sep_token_id = tokenizer.sep_token_id  # 102
pad_token_id = tokenizer.pad_token_id  # 0
mask_token_id = tokenizer.mask_token_id  # 103
```

---

## PERFORMANCE OPTIMIZATION TIPS

### 1. Use Batch Processing (NOT Sequential Loops)

**❌ DON'T (Slow - 10+ seconds for 1000 texts):**
```python
tokens = []
for text in texts:  # Sequential processing
    token_ids = bert_tokenizer.tokenize(text)
    tokens.append(token_ids)
```

**✅ DO (Fast - ~100ms for 1000 texts):**
```python
# Vectorized batch processing
tokens = bert_tokenizer.tokenize(texts)  # All at once with tf.data
```

### 2. Pre-compute Vocabulary Lookup Tables (NOT On-the-fly)

**❌ DON'T (Slow - rebuilds table each time):**
```python
for batch in data:
    vocab_table = tf.lookup.StaticVocabularyTable(vocab, num_oov_buckets=1)
    vocab_table.lookup(tokens)  # Table rebuilt for every batch!
```

**✅ DO (Fast - build once, reuse):**
```python
vocab_table = tf.lookup.StaticVocabularyTable(vocab, num_oov_buckets=1)
# In loop:
for batch in data:
    token_ids = vocab_table.lookup(tokens)  # Reuse same table
```

### 3. Use Ragged Tensors for Variable Lengths (NOT Dense with Padding)

**❌ DON'T (Memory waste for short sequences):**
```python
# If max_length=512, every sequence padded to 512
padded = tf.pad(tokens, [[0,0], [0, 512-len(tokens)]])
# 100-token sequence wastes 412 tokens of memory per item!
```

**✅ DO (Memory efficient):**
```python
# Keep variable length with RaggedTensor
ragged_tokens = tf.ragged.constant(tokens)
# Process ragged directly or convert to dense only when needed
padded_dense = ragged_tokens.to_tensor(shape=[None, 512])
```

### 4. Apply Trimming BEFORE Padding (NOT After)

**❌ DON'T (Slow - pads then trims):**
```python
padded = pad_model_inputs(tokens, max_length=512)  # Pads to 512
trimmed = trim_model_inputs(padded, max_length=128)  # Then trims to 128!
```

**✅ DO (Fast - trim once, then pad):**
```python
trimmed = tf.text.trim_model_inputs(tokens, max_length=128)
padded = pad_model_inputs(trimmed, max_length=128)
```

### 5. Use tf.data.Dataset Pipeline (NOT Manual Iteration)

**❌ DON'T (No optimization, CPU bottleneck):**
```python
for texts in manual_batches:
    tokens = tokenize(texts)  # Tokenization happens on main thread
    model_output = model(tokens)
```

**✅ DO (Optimized - prefetching, parallel processing):**
```python
dataset = tf.data.Dataset.from_tensor_slices(texts) \
    .batch(32) \
    .map(tokenize_batch, num_parallel_calls=tf.data.AUTOTUNE) \
    .prefetch(tf.data.AUTOTUNE)
    
for tokens in dataset:
    model_output = model(tokens)  # GPU/TPU has data ready before compute
```

### 6. Cache Normalized Text When Reused

**❌ DON'T (Normalize every epoch):**
```python
for epoch in range(100):
    for text in texts:
        normalized = tf_text.case_fold_utf8(text)  # Recalculate every epoch!
        tokens = tokenize(normalized)
```

**✅ DO (Normalize once, cache):**
```python
# Preprocess once, save to disk
normalized_texts = tf_text.case_fold_utf8(texts)
dataset = tf.data.Dataset.from_tensor_slices(normalized_texts) \
    .batch(32) \
    .cache()  # Cache in memory after first epoch
    
for epoch in range(100):
    dataset = dataset.shuffle(1000)
    for text in dataset:
        tokens = tokenize(text)  # No re-normalization!
```

---

## DO's AND DON'Ts WITH CODE EXAMPLES

### 1. Vocabulary Management

**DON'T: Pass TensorFlow Tensor as Vocabulary**
```python
# ❌ WRONG - Raises error
vocab_tensor = tf.constant(['the', 'cat', 'sat'])
tokenizer = tf_text.WordpieceTokenizer(vocab=vocab_tensor)
# Error: Expected Python list, got Tensor
```

**DO: Pass Python List as Vocabulary**
```python
# ✅ CORRECT
vocab_list = ['the', 'cat', 'sat', '[UNK]']
tokenizer = tf_text.FastWordpieceTokenizer(vocab=vocab_list)
tokens = tokenizer.tokenize(['the cat sat'])
# Output: [[1, 2, 3]] (token IDs)
```

### 2. Handling Unknown Words (OOV)

**DON'T: Forget num_oov_buckets in lookup table**
```python
# ❌ WRONG - Crashes on unknown word
vocab = tf.lookup.KeyValueTensorInitializer(
    keys=tf.constant(['the', 'cat']),
    values=tf.range(len(['the', 'cat']), dtype=tf.int64)
)
table = tf.lookup.StaticHashTable(vocab, default_value=-1)
# Unknown word 'dog' -> -1, causes indexing errors in embedding layer
```

**DO: Use num_oov_buckets for safe OOV handling**
```python
# ✅ CORRECT - Safe OOV handling
vocab = tf.lookup.KeyValueTensorInitializer(
    keys=tf.constant(['the', 'cat']),
    values=tf.range(2, dtype=tf.int64)
)
table = tf.lookup.StaticVocabularyTable(vocab, num_oov_buckets=1)
# Unknown word 'dog' -> id=3 (mapped to OOV bucket automatically)
```

### 3. Multi-Segment Inputs (BERT Style)

**DON'T: Forget token_type_ids for multi-segment model**
```python
# ❌ WRONG - Model ignores which segment is which
text_a = [101, 2054, 2003, 102]     # [CLS] + "what is" + [SEP]
text_b = [2009, 1045, 2572, 102]    # "so I wonder" + [SEP]
combined = text_a + text_b
model_output = model(input_ids=combined)
# Model doesn't know text_b follows text_a!
```

**DO: Include token_type_ids to distinguish segments**
```python
# ✅ CORRECT - Model knows segment boundaries
text_a = [101, 2054, 2003, 102]
text_b = [2009, 1045, 2572, 102]
combined = text_a + text_b  # [101, 2054, 2003, 102, 2009, 1045, 2572, 102]

token_type_ids = [0, 0, 0, 0,      # Segment A = 0
                  1, 1, 1, 1]      # Segment B = 1

attention_mask = [1, 1, 1, 1, 1, 1, 1, 1]  # All real tokens

output = model(input_ids=combined, 
               token_type_ids=token_type_ids,
               attention_mask=attention_mask)
```

### 4. Order of Operations: Normalize → Tokenize → Lookup

**DON'T: Tokenize then normalize**
```python
# ❌ WRONG - Loses Unicode normalization benefits
tokens = bert_tokenizer.tokenize(raw_text)  # Still has accents
normalized = tf_text.normalize_utf8(tokens)  # Too late!
```

**DO: Normalize before tokenization**
```python
# ✅ CORRECT - Normalize first for consistent tokenization
normalized = tf_text.normalize_utf8(raw_text, 'NFC')
tokens = bert_tokenizer.tokenize(normalized)
# Accents now properly composed, tokenizer sees consistent forms
```

### 5. Attention Masks for Padding

**DON'T: Forget attention_mask with padding**
```python
# ❌ WRONG - Model attends to padding tokens!
input_ids = [[101, 2054, 102, 0, 0, 0]]  # Padded with zeros
model_output = model(input_ids)
# Model uses padding token representations in attention
# Results in degraded output quality
```

**DO: Use attention_mask to mask padding**
```python
# ✅ CORRECT - Model ignores padding
input_ids = [[101, 2054, 102, 0, 0, 0]]
attention_mask = [[1, 1, 1, 0, 0, 0]]  # 1 = real token, 0 = padding

model_output = model(input_ids, attention_mask=attention_mask)
# Model masks attention to position [3,4,5] completely
```

### 6. SentencePiece vs WordPiece

**DON'T: Use SentencePiece for BERT model**
```python
# ❌ WRONG - Incompatible tokenizers
sp_tokenizer = tf_text.SentencepieceTokenizer(model_bytes)
tokens = sp_tokenizer.tokenize(text)  # SentencePiece IDs
bert_model = tf.keras.models.load_model('bert-base-uncased')
output = bert_model(tokens)  # Token IDs don't match BERT vocab!
```

**DO: Match tokenizer to model**
```python
# ✅ CORRECT - Use correct tokenizer
# For BERT models:
bert_tokenizer = tf_text.BertTokenizer(lower_case=True)
tokens = bert_tokenizer.tokenize(text)
bert_model = tf.keras.models.load_model('bert-base-uncased')
output = bert_model(tokens)

# For SentencePiece models:
sp_tokenizer = tf_text.SentencepieceTokenizer(model_bytes)
tokens = sp_tokenizer.tokenize(text)
sp_model = tf.keras.models.load_model('sentencepiece-model')
output = sp_model(tokens)
```

### 7. Export for Production (Fast vs Regular Tokenizers)

**DON'T: Export regular SentencepieceTokenizer to SavedModel**
```python
# ❌ WRONG - Fails on export due to pywrap dependencies
sp_tokenizer = tf_text.SentencepieceTokenizer(model_bytes)
tokenizer_model = tf.keras.Sequential([sp_tokenizer])
tf.saved_model.save(tokenizer_model, '/path/to/model')
# Fails: C++ op dependencies not portable to production
```

**DO: Use Fast variant for SavedModel export**
```python
# ✅ CORRECT - Export works reliably
fast_sp_tokenizer = tf_text.FastSentencepieceTokenizer(model_bytes)
tokenizer_model = tf.keras.Sequential([fast_sp_tokenizer])
tf.saved_model.save(tokenizer_model, '/path/to/model')
# Succeeds: TFLite-compatible, no external dependencies
```

---

## TIPS & TRICKS WITH EXAMPLES

### Tip 1: Inspect RaggedTensor Structure Without Full Materialization

```python
tokens = tf_text.tokenize(texts)  # Returns RaggedTensor

# DON'T do this (loads everything to memory):
# full_tensor = tokens.numpy()  # Huge memory usage!

# DO this instead (efficient inspection):
print(f"Number of sequences: {tokens.nrows()}")
print(f"Tokens per sequence: {tokens.row_lengths().numpy()}")
print(f"Total tokens: {len(tokens.flat_values)}")
print(f"First sequence: {tokens[0].numpy()}")
# Example output:
# Number of sequences: 32
# Tokens per sequence: [8, 12, 10, ...]
# Total tokens: 352
# First sequence: [101 2054 2003 102]
```

### Tip 2: Use wordshape Features for Free NER Boost

```python
# Add character pattern features to tokens for better NER
tokens = tf_text.tokenize(text)
word_shapes = tf_text.wordshape(tokens, word_shape_pattern='title_case|all_caps|lower|upper|mixed')

# Example output:
# Original tokens: ['John', 'Smith', 'ACME', 'Inc.', 'is', 'here']
# word_shapes:     [True,  True,    True,   False, False, False]
# (True = Title/Caps, False = lowercase/mixed)

# Use this as additional feature channel:
token_embeddings = embedding_layer(tokens)  # [batch, seq, emb_dim]
shape_embeddings = shape_embedding_layer(word_shapes)  # [batch, seq, 8]
combined = tf.concat([token_embeddings, shape_embeddings], axis=-1)
# Now model has character-level signal for free!
```

### Tip 3: Overlap Chunking for Long Sequences (Avoid Cutting Entities)

```python
long_text = "John Smith works at ACME Corp. He is CEO. Alice Johnson also works there."
tokens = tf_text.tokenize(long_text)  # 20 tokens

# DON'T use adjacent windows (cuts entities):
# chunks = [tokens[0:8], tokens[8:16], tokens[16:20]]
# Might cut "ACME Corp" across chunk boundary!

# DO use overlapping windows:
window_width = 8
stride = 6  # Overlap of 2 tokens
num_chunks = (len(tokens) - window_width) // stride + 1

overlapped_chunks = []
for i in range(num_chunks):
    start = i * stride
    end = min(start + window_width, len(tokens))
    chunk = tokens[start:end]
    overlapped_chunks.append(chunk)

# Example: chunks overlap by 2 tokens
# Chunk 0: tokens[0:8]   (overlap on right)
# Chunk 1: tokens[6:14]  (overlap on both sides)
# Chunk 2: tokens[12:20] (overlap on left)
# Now entity "ACME Corp" appears complete in at least one chunk!
```

### Tip 4: Cache Computationally Expensive Operations

```python
# For normalize_utf8 (relatively expensive with large texts):
raw_texts = tf.constant(['Café', 'Naïve', 'Résumé'] * 1000)

# DON'T normalize repeatedly:
# for epoch in range(100):
#     normalized = tf_text.normalize_utf8(raw_texts)  # Recalculates!

# DO normalize once and cache:
normalized = tf_text.normalize_utf8(raw_texts, 'NFKC')

dataset = tf.data.Dataset.from_tensor_slices(normalized) \
    .batch(32) \
    .cache('cache_dir') \
    .repeat(100)

for text_batch in dataset:
    tokens = tokenize(text_batch)
    # normalized text loaded from cache, not recalculated
```

### Tip 5: Efficient Token-to-Text Mapping (For Error Analysis)

```python
# Keep token_ids and original_text aligned for debugging NER errors
tokens = tf_text.tokenize(text)
token_ids = vocab_table.lookup(tokens)

# For every prediction, trace back to original text:
prediction_debug = []
for i, (token_id, token_text) in enumerate(zip(token_ids, tokens)):
    prediction_debug.append({
        'id': token_id,
        'text': token_text.numpy().decode('utf-8'),
        'prediction': predicted_labels[i]
    })

# Example output:
# [{'id': 2054, 'text': 'what', 'prediction': 'O'},
#  {'id': 2003, 'text': 'is', 'prediction': 'O'},
#  ...]
# Now easily debug why prediction is wrong!
```

### Tip 6: Use FirstNItemSelector for MLM Token Selection (Efficient)

```python
tokens = tf_text.tokenize(text)  # [batch, seq_len]
num_to_mask = 0.15 * seq_len  # Mask 15% for MLM

# DON'T use random sampling (inefficient for large batches):
# random_indices = tf.random.shuffle(tf.range(seq_len))[:num_to_mask]

# DO use FirstNItemSelector (vectorized):
selector = tf_text.FirstNItemSelector(n=int(num_to_mask))
mask = selector.get_selection_mask(tokens)  # Boolean mask [batch, seq_len]

# Create [MASK] tokens efficiently:
masked_tokens = tf.where(mask, MASK_TOKEN, tokens)
# Much faster for large batches!
```

### Tip 7: Debug Ragged Tensor Shapes

```python
tokens = tf_text.tokenize([
    'hello world',
    'foo bar baz',
    'a',
])

print(f"Shape: {tokens.shape}")  # (None, None) - not helpful!
print(f"Rank: {tokens.ragged_rank}")  # 1 - one level of raggedness

# DO use row_lengths for debugging:
print(f"Tokens per text: {tokens.row_lengths().numpy()}")
# Output: [2 3 1]
# Text 1: 2 tokens, Text 2: 3 tokens, Text 3: 1 token

print(f"Max tokens: {tf.reduce_max(tokens.row_lengths())}")  # 3
print(f"Total tokens: {len(tokens.flat_values)}")  # 6

# This helps verify tokenization worked correctly!
```

### Tip 8: Handle Unicode Normalization for Consistency Across Languages

```python
# Different languages need different normalization forms
texts = tf.constant(['café', 'Ⅲ', 'ﬁle'])  # Precomposed, fullwidth, ligatures

# For Western languages (NFC - composed):
nfc = tf_text.normalize_utf8(texts, 'NFC')
# Output: 'café' (é as single codepoint)

# For compatibility (NFKC - decomposed + compatibility):
nfkc = tf_text.normalize_utf8(texts, 'NFKC')
# Output: 'cafe' (é decomposed to e + accent)
#         'III' (fullwidth to ASCII)
#         'file' (ligatures split)

# Use based on your model needs:
# NFC: Preserve visual similarity, smaller vocab size
# NFKC: More consistent across variants, helps with misspellings
for normalization_form in ['NFC', 'NFD', 'NFKC', 'NFKD']:
    result = tf_text.normalize_utf8(texts, normalization_form)
    print(f"{normalization_form}: {result.numpy()}")
```

In [40]:
# Tip 6: Word Shape Features for NER
print("\n=== TIP 6: Word Shape Features (Free NER Boost) ===")
test_words = ws_tokenizer.tokenize(tf.constant(['John ACME the Inc. email123']))[0]
try:
    # wordshape detects character patterns
    print(f"Words: {[w.decode('utf-8') for w in test_words.numpy()]}")
    print("Word shape patterns:")
    for word in test_words.numpy():
        word_str = word.decode('utf-8')
        # Detect shape manually
        if word_str.isupper():
            shape = 'ALL_UPPER'
        elif word_str[0].isupper():
            shape = 'Title_Case'
        elif word_str.islower():
            shape = 'all_lower'
        else:
            shape = 'MiXeD'
        print(f"  '{word_str:12}' → {shape}")
except Exception as e:
    print(f"  Note: wordshape API varies - manual detection shown above")


=== TIP 6: Word Shape Features (Free NER Boost) ===
Words: ['John', 'ACME', 'the', 'Inc.', 'email123']
Word shape patterns:
  'John        ' → Title_Case
  'ACME        ' → ALL_UPPER
  'the         ' → all_lower
  'Inc.        ' → Title_Case
  'email123    ' → all_lower
